# Cell–cell communication (CCC) by sex with **cell2cell** (mouse dataset)

This notebook is designed to run **after** you have an `AnnData` object named `adata` in memory (Scanpy workflow).
It is **memory-safe** for large datasets by working on:
- a **mouse** ligand–receptor list (to avoid symbol mismatch),
- **pseudobulk means** per *donor × cell type* for inference/testing,
- and **sex-aggregated** cell-type profiles for the main cell2cell visualizations.


In [ ]:
# =========================
# 0) Imports + settings
# =========================
import os
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy import stats

import scanpy as sc
import matplotlib.pyplot as plt

import cell2cell as c2c

# Optional (used only for a couple of nicer plots/heatmaps)
try:
    import seaborn as sns
    _HAS_SNS = True
except Exception:
    _HAS_SNS = False

import warnings
warnings.filterwarnings("ignore")

RESULTS_DIR = Path("ccc_by_sex_cell2cell")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(0)
plt.rcParams["figure.dpi"] = 120

print("cell2cell:", getattr(c2c, "__version__", "unknown"))
print("scanpy:", sc.__version__)


In [ ]:
import os
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy import sparse

warnings.filterwarnings("ignore")

# Enable inline plotting (Jupyter)
%matplotlib inline

# -----------------------------
# CONFIG
# -----------------------------
DATA_PATH = Path("Soni_Brain_2025_cellxgene_5a6c74b9-da1c-4cef-8fdc-07c7a90089d7.h5ad")
RESULTS_DIR = Path("sex_analysis_results")
RESULTS_DIR.mkdir(exist_ok=True, parents=True)

# Candidate keys (auto-detection)
SEX_KEY_CANDIDATES = ["sex", "Sex", "gender", "Gender"]
CELLTYPE_KEY_CANDIDATES = ["cell_type", "celltype", "CellType", "annotation", "celltype_major"]
SAMPLE_KEY_CANDIDATES = ["sample", "sample_id", "sampleID", "donor_id", "donor", "patient", "orig.ident", "library_id"]

# Plotting controls
PLOT_TOP_CELLTYPES = 12               # for faceted violin plots
PLOT_MAX_CELLS_PER_GROUP = 500        # downsample for violin plots (per celltype x sex)
HEATMAP_TOP_GENES = 30                # genes per heatmap
PSEUDOBULK_USE_LAYER = "counts"       # prefer raw counts if available
PSEUDOBULK_TARGET_SUM = 1e6           # CPM-like
DE_MIN_SAMPLES_PER_GROUP = 2          # for sample-level DE

# Optional: specify sample groups manually (example)
# SAMPLE_GROUPS = {"female": ["X1", "X2"], "male": ["X3", "X4"]}
SAMPLE_GROUPS = None

# CNV controls (InferCNVpy)
RUN_CNV = True
CNV_DOWNSAMPLE_EVERY_NTH_CELL = 5     # increase to speed up (e.g. 10, 20) if needed
CNV_WINDOW_SIZE = 200
CNV_STEP = 25
CNV_REFERENCE_KEY = None              # e.g. "cell_type" if you provide reference categories
CNV_REFERENCE_CATS = None             # e.g. ["T cell", "B cell", ...] (normal/reference)
CNV_EXCLUDE_CHROMS = ("chrX", "chrY") # avoid sex chr confounding (default)

# Enrichment controls (fast ORA)
RUN_ENRICHMENT = True
ENRICHR_LIBRARY = "Hallmark_2020"     # small & fast; change if needed
ENRICHMENT_MAX_GENES = 200            # limit query list
ENRICHMENT_P_ADJ_CUTOFF = 0.05
ENRICHMENT_MIN_OVERLAP = 3

# Optional: fast pre-ranked GSEA (still heavier than ORA)
RUN_PRERANK_GSEA = False
GSEA_TOP_CELLTYPES = 6
GSEA_PERMUTATIONS = 200

np.random.seed(0)

# -----------------------------
# Helpers
# -----------------------------
def pick_obs_key(adata: sc.AnnData, candidates: List[str], required: bool = True) -> Optional[str]:
    # Return the first matching obs column name from candidates.
    for k in candidates:
        if k in adata.obs.columns:
            return k
    if required:
        raise KeyError(
            f"None of the candidate keys were found in adata.obs: {candidates}. "
            f"Available keys (first 50): {list(adata.obs.columns)[:50]}"
        )
    return None

def standardize_sex_series(s: pd.Series) -> pd.Series:
    # Map common sex/gender encodings to {'female','male'} when possible.
    x = s.astype(str).str.strip()
    xl = x.str.lower()
    mapping = {
        "f": "female", "female": "female", "woman": "female", "w": "female", "0": "female",
        "m": "male", "male": "male", "man": "male", "1": "male",
    }
    return xl.map(lambda v: mapping.get(v, v))

def bh_fdr(pvals: np.ndarray) -> np.ndarray:
    # Benjamini-Hochberg FDR correction.
    p = np.asarray(pvals, dtype=float)
    n = p.size
    order = np.argsort(p)
    ranked = np.empty(n, dtype=float)
    ranked[order] = np.arange(1, n + 1)
    q = p * n / ranked
    q_ordered = q[order]
    q_ordered = np.minimum.accumulate(q_ordered[::-1])[::-1]
    q_adj = np.empty(n, dtype=float)
    q_adj[order] = np.clip(q_ordered, 0, 1)
    return q_adj

def downsample_obs(df: pd.DataFrame, group_cols: List[str], max_per_group: int, seed: int = 0) -> pd.DataFrame:
    # Downsample rows in a dataframe to max_per_group per group.
    if max_per_group is None or max_per_group <= 0:
        return df
    return (
        df.groupby(group_cols, group_keys=False)
          .apply(lambda g: g.sample(n=min(len(g), max_per_group), random_state=seed))
          .reset_index(drop=True)
    )

def pseudobulk_sum_counts(adata: sc.AnnData, group_key: str, layer: Optional[str] = "counts") -> Tuple[pd.DataFrame, pd.Series]:
    # Pseudobulk sum counts per group_key.
    if layer is not None and layer in adata.layers:
        X = adata.layers[layer]
    else:
        X = adata.X
    if sparse.issparse(X):
        X = X.tocsr()
    else:
        X = np.asarray(X)

    groups = adata.obs[group_key].astype(str).values
    cats, inv = np.unique(groups, return_inverse=True)
    n_groups = len(cats)
    n_cells = adata.n_obs

    G = sparse.csr_matrix(
        (np.ones(n_cells, dtype=np.float32), (inv, np.arange(n_cells))),
        shape=(n_groups, n_cells),
    )
    sums = G @ X  # (n_groups x n_genes)
    if sparse.issparse(sums):
        sums = sums.toarray()
    sums_df = pd.DataFrame(sums, index=cats, columns=adata.var_names)
    libsize = sums_df.sum(axis=1)
    return sums_df, libsize

def pseudobulk_logcpm(adata: sc.AnnData, group_key: str, layer: Optional[str] = "counts", target_sum: float = 1e6) -> pd.DataFrame:
    # Pseudobulk to log1p(CPM) per group.
    sums_df, libsize = pseudobulk_sum_counts(adata, group_key=group_key, layer=layer)
    cpm = sums_df.div(libsize.replace(0, np.nan), axis=0) * target_sum
    cpm = cpm.fillna(0.0)
    return np.log1p(cpm)

def sample_sex_map(adata: sc.AnnData, sample_key: str, sex_key: str) -> pd.Series:
    # Infer sex per sample by majority vote.
    tmp = adata.obs[[sample_key, sex_key]].copy()
    tmp[sample_key] = tmp[sample_key].astype(str)
    tmp[sex_key] = tmp[sex_key].astype(str)

    def majority(x: pd.Series) -> str:
        vc = x.value_counts()
        return vc.index[0] if len(vc) else np.nan

    return tmp.groupby(sample_key)[sex_key].apply(majority)

def de_ttest_pseudobulk(pb: pd.DataFrame, group_labels: pd.Series, group_a: str, group_b: str) -> pd.DataFrame:
    # Differential expression on pseudo-bulk matrix (samples x genes).
    group_labels = group_labels.loc[pb.index]
    a_idx = group_labels[group_labels == group_a].index
    b_idx = group_labels[group_labels == group_b].index
    A = pb.loc[a_idx]
    B = pb.loc[b_idx]

    meanA = A.mean(axis=0)
    meanB = B.mean(axis=0)

    cpmA = np.expm1(meanA)
    cpmB = np.expm1(meanB)
    log2fc = np.log2((cpmA + 1e-6) / (cpmB + 1e-6))

    _, pvals = stats.ttest_ind(A.values, B.values, axis=0, equal_var=False, nan_policy="omit")
    padj = bh_fdr(pvals)

    out = pd.DataFrame(
        {
            "gene": pb.columns,
            "log2fc": log2fc.values,
            "pval": pvals,
            "padj": padj,
            "mean_logcpm_a": meanA.values,
            "mean_logcpm_b": meanB.values,
            "n_a": len(a_idx),
            "n_b": len(b_idx),
        }
    ).sort_values(["padj", "pval"])
    return out

print("=" * 80)
print("SEX-BASED DIFFERENTIAL EXPRESSION ANALYSIS — UPDATED NOTEBOOK")
print("=" * 80)

# scanpy settings
sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=120, facecolor="white", figsize=(8, 6))
plt.rcParams["figure.figsize"] = (10, 8)
sns.set_style("whitegrid")


In [ ]:
print("\n[1/12] Loading data...")
print(f"Reading: {DATA_PATH.resolve()}")
adata = sc.read_h5ad(str(DATA_PATH))

print(f"Dataset shape: {adata.shape}")
print(f"Available obs keys (first 30): {list(adata.obs.columns)[:30]}")
print(f"Available embeddings: {list(adata.obsm.keys())}")

# Auto-detect key columns
sex_key = pick_obs_key(adata, SEX_KEY_CANDIDATES, required=True)
celltype_key = pick_obs_key(adata, CELLTYPE_KEY_CANDIDATES, required=True)
sample_key = pick_obs_key(adata, SAMPLE_KEY_CANDIDATES, required=False)

# Standardize sex to {female,male} when possible
adata.obs[sex_key] = standardize_sex_series(adata.obs[sex_key])

sex_counts = adata.obs[sex_key].value_counts(dropna=False)
print("\nSex distribution (standardized):")
print(sex_counts)

if ("female" in sex_counts.index) and ("male" in sex_counts.index):
    female_label, male_label = "female", "male"
else:
    top2 = sex_counts.index[:2].tolist()
    if len(top2) < 2:
        raise ValueError("Need at least 2 sex categories for sex-based analysis.")
    female_label, male_label = str(top2[0]), str(top2[1])
    print(f"\n[WARN] Could not confidently map to female/male. Using top-2 categories: {female_label}, {male_label}")

# Sample key fallback
if sample_key is None:
    sample_key = "donor_id" if "donor_id" in adata.obs.columns else None
if sample_key is None:
    raise KeyError("Could not find a sample/donor key. Please set SAMPLE_KEY_CANDIDATES or add a sample column.")

print(f"\nDetected keys:\n  sex_key={sex_key}\n  celltype_key={celltype_key}\n  sample_key={sample_key}")
print(f"Cell types: {adata.obs[celltype_key].nunique()}")
print(f"Samples: {adata.obs[sample_key].nunique()}")

# Clean types
adata.obs[celltype_key] = adata.obs[celltype_key].astype(str)
adata.obs[sample_key] = adata.obs[sample_key].astype(str)
adata.obs[sex_key] = adata.obs[sex_key].astype("category")


In [ ]:
# ============================================
# CCC by sex (mouse) with cell2cell-style LR
# - robust LR parsing / mapping (fix mapped_pairs=0)
# - sparse pseudobulk (no densification)
# - sex-differential edges (soft correction)
# - publication-ready plots
# ============================================

import re
import os
import math
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import sparse, stats

# ---- Optional (only used for plotting styles) ----
try:
    import cell2cell as c2c
    HAVE_CELL2CELL = True
except Exception as e:
    HAVE_CELL2CELL = False
    print("[WARN] cell2cell not importable:", repr(e))

# -----------------------------
# CONFIG
# -----------------------------
RESULTS_DIR = Path("ccc_by_sex_cell2cellstyle")
RESULTS_DIR.mkdir(exist_ok=True, parents=True)

# LR source (mouse). This file has ligand_symbol / receptor_symbol in mouse-style casing.
LR_URL = "https://raw.githubusercontent.com/LewisLabUCSD/Ligand-Receptor-Pairs/master/Mouse/Mouse-2020-Jin-LR-pairs.csv"
LR_CACHE = RESULTS_DIR / "Mouse-2020-Jin-LR-pairs.csv"

# expression source
PREFER_LAYER = None           # set to "lognorm" or "counts" if you know; None = auto
TARGET_SUM = 1e4              # only used if we must normalize counts subset
MIN_CELLS_PER_GROUP = 25      # drop donor×celltype groups with too few cells (stability)
MIN_DONOR_CELLS = 500         # drop donors with too few total cells
MAX_CELLTYPES = None          # e.g. 25 to restrict; None keeps all

# differential testing / softness controls
ACTIVE_EDGE_QUANTILE = 0.50   # lower => more “active edges” tested (softer / more exploratory)
STOREY_LAMBDA = 0.50
Q_CUTOFF = 0.25               # exploratory q-value threshold
TOP_EDGES_TO_PRINT = 5
TOP_EDGES_TO_PLOT = 30

np.random.seed(0)
sns.set_style("whitegrid")
sc.settings.set_figure_params(dpi=140, facecolor="white")

# -----------------------------
# Helpers
# -----------------------------
def _canon_gene(x: str) -> str:
    """Light canonicalization: strip spaces, remove Ensembl version suffix, keep digits, uppercase."""
    if x is None:
        return ""
    s = str(x).strip()
    # drop common wrappers
    s = re.sub(r"^gene[:=]\s*", "", s, flags=re.IGNORECASE)
    # remove version suffix (ENSMUSG... .12)
    s = s.split(".")[0]
    # remove parentheses content if it is annotation, e.g., "Tgfb1 (foo)"
    s = re.sub(r"\s*\(.*\)\s*", "", s)
    return s.upper()

_SPLIT_RE = re.compile(r"[&\+\|,; ]+")

def _split_complex(s: str):
    """Split complex strings into gene tokens; handles & + | , ; whitespace."""
    if s is None:
        return []
    s = str(s).strip()
    if s == "" or s.lower() == "nan":
        return []
    # normalize separators to spaces then split
    parts = [p for p in _SPLIT_RE.split(s) if p and p.lower() != "nan"]
    return parts

def _standardize_complex_to_amp(parts):
    """Join parts with '&' for cell2cell-style complexes."""
    return "&".join(parts)

def _detect_gene_namespace(adata):
    # choose a gene-name series to match LR lists
    candidates = []
    if hasattr(adata, "var") and isinstance(adata.var, pd.DataFrame):
        for c in ["gene_symbols", "feature_name", "symbol", "gene", "Gene"]:
            if c in adata.var.columns:
                candidates.append(c)

    # always include var_names
    # score overlap later
    sources = {"var_names": pd.Series(adata.var_names.astype(str), index=adata.var_names)}
    for c in candidates:
        sources[f"var[{c}]"] = adata.var[c].astype(str)

    return sources

def _build_gene_to_idx(adata, gene_series: pd.Series):
    """Map canonical gene name -> var index (first occurrence wins)."""
    gene_to_idx = {}
    for i, g in enumerate(gene_series.values):
        cg = _canon_gene(g)
        if cg and cg not in gene_to_idx:
            gene_to_idx[cg] = i
    return gene_to_idx

def _pick_expression_matrix(adata, prefer_layer=None):
    """
    Choose an expression matrix accessor without densifying.
    Returns (X_accessor, kind_str, needs_counts_norm_bool)
    - If returns counts and needs_counts_norm_bool=True, we will create lognorm for subset genes.
    """
    # If user forces a layer
    if prefer_layer is not None:
        if prefer_layer == "X":
            return ("X", "adata.X", False)
        if prefer_layer in adata.layers:
            # assume already suitable (e.g., lognorm)
            return (("layer", prefer_layer), f"adata.layers['{prefer_layer}']", False)
        raise KeyError(f"Requested layer '{prefer_layer}' not found in adata.layers")

    # Auto: prefer a log-normalized looking layer
    for k in ["lognorm", "log1p", "normalized", "X_lognorm"]:
        if k in adata.layers:
            return (("layer", k), f"adata.layers['{k}']", False)

    # Otherwise, fall back to X (could be lognorm already)
    # Heuristic: if X has negatives, it's scaled => try counts
    X = adata.X
    try:
        mn = X.min() if not sparse.issparse(X) else X.min()
        mn = float(mn)
    except Exception:
        mn = 0.0

    if mn < -1e-6:
        # scaled data -> try counts
        if "counts" in adata.layers:
            return (("layer", "counts"), "adata.layers['counts'] (will log-normalize subset)", True)
        else:
            print("[WARN] adata.X looks scaled (negative values) and no counts layer found; using adata.X anyway.")
            return ("X", "adata.X", False)

    return ("X", "adata.X", False)

def _get_X(adata, X_spec):
    if X_spec == "X":
        return adata.X
    kind, name = X_spec
    assert kind == "layer"
    return adata.layers[name]

def _sparse_group_mean(X_csr, group_labels: np.ndarray, min_cells=1):
    """
    Compute group means for sparse matrix X (cells x genes) given group_labels (len=cells).
    Returns:
      means: (n_groups x n_genes) dense float32
      groups: array of group names
      counts: array of group sizes
    """
    group_labels = np.asarray(group_labels).astype(str)
    groups, inv = np.unique(group_labels, return_inverse=True)
    n_groups = len(groups)
    n_cells = X_csr.shape[0]

    # group membership matrix G (n_groups x n_cells), nnz = n_cells
    G = sparse.csr_matrix(
        (np.ones(n_cells, dtype=np.float32), (inv, np.arange(n_cells))),
        shape=(n_groups, n_cells)
    )
    sums = (G @ X_csr).astype(np.float32)  # (groups x genes)
    if sparse.issparse(sums):
        sums = sums.toarray()
    counts = np.asarray(G.sum(axis=1)).reshape(-1).astype(np.float32)
    means = sums / np.maximum(counts[:, None], 1.0)

    # filter min_cells
    keep = counts >= float(min_cells)
    return means[keep], groups[keep], counts[keep]

def storey_qvalues(pvals, lam=0.5):
    """
    Storey-Tibshirani q-values (simple pi0 estimate).
    """
    p = np.asarray(pvals, dtype=float)
    p = np.clip(p, 0, 1)
    m = p.size
    if m == 0:
        return p

    # pi0 estimate
    pi0 = np.mean(p > lam) / max(1e-8, (1.0 - lam))
    pi0 = float(np.clip(pi0, 0, 1))

    # BH-like with pi0 scaling
    order = np.argsort(p)
    ranked = np.arange(1, m + 1, dtype=float)
    q = pi0 * p[order] * m / ranked
    q = np.minimum.accumulate(q[::-1])[::-1]
    out = np.empty_like(q)
    out[order] = np.clip(q, 0, 1)
    return out

# -----------------------------
# 0) Ensure sex labels are real female/male (handles PATO ids)
# -----------------------------
# Common in cellxgene exports:
# PATO:0000383 = female, PATO:0000384 = male
if "sex_ontology_term_id" in adata.obs.columns:
    # if sex is missing or single-valued, recover from ontology
    if adata.obs.get("sex", pd.Series(index=adata.obs_names, dtype=str)).nunique() <= 1:
        pato = adata.obs["sex_ontology_term_id"].astype(str)
        map_pato = {"PATO:0000383": "female", "PATO:0000384": "male"}
        adata.obs["sex"] = pato.map(map_pato).fillna(pato)
        print("[INFO] Recovered sex labels from sex_ontology_term_id.")

# If you already have sex_key/celltype_key/sample_key from your earlier cells, keep them.
# Otherwise, infer simple defaults:
sex_key = globals().get("sex_key", "sex")
celltype_key = globals().get("celltype_key", "cell_type" if "cell_type" in adata.obs.columns else "sub_celltype")
sample_key = globals().get("sample_key", "donor_id" if "donor_id" in adata.obs.columns else "sample")

adata.obs[sex_key] = adata.obs[sex_key].astype(str).str.strip().str.lower()
# normalize common variants
adata.obs[sex_key] = adata.obs[sex_key].replace({"f":"female","m":"male","woman":"female","man":"male"})

print("\n[INFO] Sex counts:")
print(adata.obs[sex_key].value_counts(dropna=False))

# -----------------------------
# 1) Load LR pairs (mouse) + robust parsing
# -----------------------------
print("\n[1] Loading LR pairs...")
if not LR_CACHE.exists():
    lr_raw = pd.read_csv(LR_URL)
    lr_raw.to_csv(LR_CACHE, index=False)
    print(f"[OK] Downloaded LR pairs -> {LR_CACHE} | rows={len(lr_raw)}")
else:
    lr_raw = pd.read_csv(LR_CACHE)
    print(f"[OK] Loaded LR pairs from cache -> {LR_CACHE} | rows={len(lr_raw)}")

# Pick columns robustly
cols = {c.lower(): c for c in lr_raw.columns}
def pick_col(*names):
    for n in names:
        if n.lower() in cols:
            return cols[n.lower()]
    return None

lig_col = pick_col("ligand_symbol", "ligand", "ligand_gene", "ligandname", "ligand_name")
rec_col = pick_col("receptor_symbol", "receptor", "receptor_gene", "receptorname", "receptor_name")

if lig_col is None or rec_col is None:
    raise RuntimeError(f"Could not find ligand/receptor columns in LR file. Columns={list(lr_raw.columns)}")

lr_df = lr_raw[[lig_col, rec_col]].copy()
lr_df.columns = ["ligand_raw", "receptor_raw"]
lr_df = lr_df.dropna()

# Expand complexes safely and standardize to '&'
pairs = []
for L, R in lr_df[["ligand_raw","receptor_raw"]].itertuples(index=False):
    L_parts = _split_complex(L)
    R_parts = _split_complex(R)
    if len(L_parts) == 0 or len(R_parts) == 0:
        continue
    # Usually ligand is a single gene; if multiple, expand
    for l in L_parts:
        pairs.append((_standardize_complex_to_amp([l]), _standardize_complex_to_amp(R_parts)))

pairs_df = pd.DataFrame(pairs, columns=["ligand_symbol", "receptor_symbol"]).drop_duplicates()
print(f"[INFO] Parsed LR pairs: {len(pairs_df)} | unique ligands={pairs_df['ligand_symbol'].nunique()} | unique receptors={pairs_df['receptor_symbol'].nunique()}")

# -----------------------------
# 2) Pick best gene namespace in adata (max overlap with LR genes)
# -----------------------------
print("\n[2] Resolving gene namespace overlap...")
lr_genes = set()
for l, r in pairs_df[["ligand_symbol","receptor_symbol"]].itertuples(index=False):
    for g in _split_complex(l): lr_genes.add(_canon_gene(g))
    for g in _split_complex(r): lr_genes.add(_canon_gene(g))

sources = _detect_gene_namespace(adata)

best_name, best_overlap, best_series = None, -1, None
for name, ser in sources.items():
    gene_set = set(_canon_gene(x) for x in ser.values)
    ov = len(gene_set.intersection(lr_genes))
    print(f"  - {name}: overlap={ov}")
    if ov > best_overlap:
        best_overlap, best_name, best_series = ov, name, ser

print(f"[INFO] Selected gene names from: {best_name} | overlap={best_overlap}")

if best_overlap == 0:
    # dump diagnostics
    print("[DIAG] Example adata genes:", list(best_series.values[:20]))
    print("[DIAG] Example LR genes:", list(sorted(list(lr_genes))[:20]))
    raise RuntimeError(
        "LR overlap is still 0. This usually means your AnnData does not store mouse gene symbols in var_names/var columns.\n"
        "Check adata.var columns for symbols/Ensembl and re-run with the right column."
    )

gene_to_idx = _build_gene_to_idx(adata, best_series)

# Now map LR pairs into adata indices (require all complex parts present)
mapped_rows = []
missing = 0
for l, r in pairs_df[["ligand_symbol","receptor_symbol"]].itertuples(index=False):
    l_parts = [_canon_gene(x) for x in _split_complex(l)]
    r_parts = [_canon_gene(x) for x in _split_complex(r)]
    if any(p not in gene_to_idx for p in l_parts):  # ligand missing
        missing += 1
        continue
    if any(p not in gene_to_idx for p in r_parts):  # receptor complex missing
        missing += 1
        continue
    # keep standardized symbols (original casing from file is OK; matching is done via indices)
    mapped_rows.append((l, r))

lr_pairs_mapped = pd.DataFrame(mapped_rows, columns=["ligand_symbol","receptor_symbol"]).drop_duplicates()
print(f"[INFO] LR pairs mapped to your genes: kept={len(lr_pairs_mapped)} | missing={missing}")

if len(lr_pairs_mapped) < 50:
    raise RuntimeError(
        f"Too few mapped LR pairs ({len(lr_pairs_mapped)}). "
        f"Something is still mismatched. Inspect best_name={best_name} and LR file columns."
    )

# Precompute gene indices for each LR pair for fast scoring
lig_idx = np.array([gene_to_idx[_canon_gene(_split_complex(l)[0])] for l in lr_pairs_mapped["ligand_symbol"]], dtype=int)

rec_idx_list = []
for r in lr_pairs_mapped["receptor_symbol"]:
    parts = [_canon_gene(x) for x in _split_complex(r)]
    rec_idx_list.append([gene_to_idx[p] for p in parts])

rec_idx_list = rec_idx_list  # list-of-lists; receptor expression will be min across components

# -----------------------------
# 3) Choose expression source WITHOUT densifying
# -----------------------------
X_spec, X_desc, needs_counts_norm = _pick_expression_matrix(adata, prefer_layer=PREFER_LAYER)
print(f"\n[3] Using expression from: {X_desc}")

X_full = _get_X(adata, X_spec)
if sparse.issparse(X_full):
    X_full = X_full.tocsr()
else:
    # make it csr anyway
    X_full = sparse.csr_matrix(np.asarray(X_full))

# To avoid huge memory, we only slice genes used by LR pairs (unique components)
need_gene_idx = sorted(set(lig_idx.tolist() + [j for sub in rec_idx_list for j in sub]))
need_gene_idx = np.array(need_gene_idx, dtype=int)
X_lr = X_full[:, need_gene_idx]  # cells x selected_genes (sparse)

# If we fell back to counts, build lognorm on-the-fly for JUST these genes (uses total_counts if present)
if needs_counts_norm:
    if "total_counts" in adata.obs.columns:
        libsize = adata.obs["total_counts"].astype(float).values
    else:
        # compute total counts from counts matrix (could be heavier)
        libsize = np.asarray(X_full.sum(axis=1)).reshape(-1).astype(float)
    scale = TARGET_SUM / np.maximum(libsize, 1.0)
    # apply scaling per row: X_lr[i,:] *= scale[i]
    X_lr = X_lr.multiply(scale[:, None])
    # log1p sparse
    X_lr.data = np.log1p(X_lr.data)
    print("[INFO] Built lognorm from counts for LR genes only (sparse).")

# Map from original var index -> position in need_gene_idx slice
idx_pos = {g: k for k, g in enumerate(need_gene_idx.tolist())}
lig_pos = np.array([idx_pos[i] for i in lig_idx], dtype=int)
rec_pos_list = [[idx_pos[i] for i in lst] for lst in rec_idx_list]


# -----------------------------
# Tunables (more permissive than before)
# -----------------------------
MIN_DONOR_CELLS_INIT = 200          # start filter; auto-relaxes if too few donors
MIN_CELLS_PER_GROUP = 10           # donor×celltype min cells (keeps more celltypes)
ACTIVE_EDGE_QUANTILE = 0.35        # softer => more edges tested
STOREY_LAMBDA = 0.50
Q_CUTOFF = 0.25
TOP_EDGES_TO_PRINT = 5
TOP_EDGES_TO_PLOT = 30

RESULTS_DIR = Path("ccc_by_sex_cell2cellstyle")
RESULTS_DIR.mkdir(exist_ok=True, parents=True)

np.random.seed(0)
sns.set_style("whitegrid")

# -----------------------------
# Helpers
# -----------------------------
def storey_qvalues(pvals, lam=0.5):
    p = np.asarray(pvals, dtype=float)
    p = np.clip(p, 0, 1)
    m = p.size
    if m == 0:
        return p
    pi0 = np.mean(p > lam) / max(1e-8, (1.0 - lam))
    pi0 = float(np.clip(pi0, 0, 1))
    order = np.argsort(p)
    ranked = np.arange(1, m + 1, dtype=float)
    q = pi0 * p[order] * m / ranked
    q = np.minimum.accumulate(q[::-1])[::-1]
    out = np.empty_like(q)
    out[order] = np.clip(q, 0, 1)
    return out

def sparse_group_mean(X_csr, labels, min_cells=1):
    labels = np.asarray(labels)
    groups, inv = np.unique(labels, return_inverse=True)
    n_groups = len(groups)
    n_cells = X_csr.shape[0]

    G = sparse.csr_matrix(
        (np.ones(n_cells, dtype=np.float32), (inv, np.arange(n_cells))),
        shape=(n_groups, n_cells)
    )
    sums = (G @ X_csr).astype(np.float32)
    if sparse.issparse(sums):
        sums = sums.toarray()
    counts = np.asarray(G.sum(axis=1)).reshape(-1).astype(np.int32)

    keep = counts >= int(min_cells)
    means = sums[keep] / np.maximum(counts[keep, None], 1)
    return means, groups[keep], counts[keep]

def choose_best_unit_key(adata, sex_key, candidates):
    """
    Pick a sample/donor key that:
      - has good non-null fraction
      - not too many uniques (not per-cell)
      - sex is mostly pure within each unit
    """
    obs = adata.obs
    best = None
    for k in candidates:
        if k is None or k not in obs.columns:
            continue
        s = obs[k]
        # non-null mask (avoid 'nan' strings)
        nn = s.notna() & (s.astype(str).str.lower() != "nan") & (s.astype(str).str.lower() != "none") & (s.astype(str).str.strip() != "")
        nonnull_frac = float(nn.mean())
        if nonnull_frac < 0.3:
            continue

        s2 = s[nn].astype(str)
        nunique = int(s2.nunique())
        if nunique < 2:
            continue
        # avoid per-cell IDs
        if nunique > min(5000, int(0.2 * adata.n_obs)):
            continue

        df = obs.loc[nn, [k, sex_key]].copy()
        df[k] = df[k].astype(str)
        df[sex_key] = df[sex_key].astype(str)

        # purity: fraction of units with <=1 unique sex
        mix = df.groupby(k)[sex_key].nunique()
        pure_frac = float((mix <= 1).mean())

        # also prefer keys with reasonable number of units
        score = 10.0 * pure_frac + nonnull_frac + np.log1p(nunique) / 5.0
        cand = dict(key=k, score=score, nunique=nunique, nonnull_frac=nonnull_frac, pure_frac=pure_frac)
        if best is None or cand["score"] > best["score"]:
            best = cand

    return best

def heatmap(df, title, outbase, vmax=None):
    plt.figure(figsize=(10, 8))
    sns.heatmap(df, square=True, cmap="viridis", vmax=vmax)
    plt.title(title, fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f"{outbase}.png", dpi=300)
    plt.savefig(RESULTS_DIR / f"{outbase}.svg")
    plt.show()

# -----------------------------
# [4] Robust unit (donor/sample) selection + filtering
# -----------------------------
print("\n[4] Building unit×celltype means with robust unit key...")

# Ensure sex is clean at cell level
adata.obs[sex_key] = adata.obs[sex_key].astype(str).str.strip().str.lower()
adata.obs[sex_key] = adata.obs[sex_key].replace({"f":"female","m":"male","woman":"female","man":"male"})

# Candidate unit keys (put your current sample_key first if defined)
unit_candidates = []
if "sample_key" in globals() and sample_key in adata.obs.columns:
    unit_candidates.append(sample_key)
for k in ["sample_id", "donor_id", "groupid", "sample", "library_id", "orig.ident"]:
    if k not in unit_candidates:
        unit_candidates.append(k)

best_unit = choose_best_unit_key(adata, sex_key, unit_candidates)
if best_unit is None:
    raise RuntimeError(f"Could not find a usable unit key among: {unit_candidates}")

unit_key = best_unit["key"]
print(f"[INFO] Selected unit_key='{unit_key}' "
      f"(nunique={best_unit['nunique']}, nonnull={best_unit['nonnull_frac']:.2f}, sex_pure={best_unit['pure_frac']:.2f})")

# Build base mask: valid unit + valid sex
unit_series = adata.obs[unit_key]
valid_unit = unit_series.notna() & (unit_series.astype(str).str.lower() != "nan") & (unit_series.astype(str).str.strip() != "")
valid_sex = adata.obs[sex_key].isin(["female", "male"])
base_mask = (valid_unit & valid_sex).values

if base_mask.sum() == 0:
    raise RuntimeError("No cells with both valid unit id and sex label after filtering.")

# Adaptive donor filtering (relax if too few units remain)
min_unit_cells = MIN_DONOR_CELLS_INIT
unit_counts = adata.obs.loc[base_mask, unit_key].astype(str).value_counts()

def n_units_ge(thr):
    return int((unit_counts >= thr).sum())

while n_units_ge(min_unit_cells) < 6 and min_unit_cells > 50:
    min_unit_cells = max(50, min_unit_cells // 2)

good_units = unit_counts[unit_counts >= min_unit_cells].index.astype(str)
print(f"[INFO] Keeping units with >= {min_unit_cells} cells: n_units={len(good_units)}")

mask = base_mask & adata.obs[unit_key].astype(str).isin(good_units).values
ad = adata[mask].copy()

# Compute unit->sex by majority vote
tmp = ad.obs[[unit_key, sex_key]].copy()
tmp[unit_key] = tmp[unit_key].astype(str)
tmp[sex_key] = tmp[sex_key].astype(str)

unit_sex = tmp.groupby(unit_key)[sex_key].agg(lambda x: x.value_counts().index[0])
print("[INFO] Units per sex:", unit_sex.value_counts().to_dict())

# If still too few units per sex, create pseudo-replicates (exploratory)
use_pseudo = False
if unit_sex.get("female", 0) < 2 or unit_sex.get("male", 0) < 2:
    use_pseudo = True
    print("[WARN] <2 units per sex. Creating pseudo-replicates within each unit (hypothesis-generating).")
    K = 4  # number of pseudo-replicates per unit
    # assign replicate ids within each unit
    rep = np.zeros(ad.n_obs, dtype=int)
    for u in unit_sex.index.astype(str):
        idx = np.where(ad.obs[unit_key].astype(str).values == u)[0]
        if len(idx) == 0:
            continue
        rep[idx] = np.random.randint(0, K, size=len(idx))
    ad.obs["_unit_rep"] = ad.obs[unit_key].astype(str) + "__rep" + rep.astype(str)
    unit_key2 = "_unit_rep"

    # recompute unit_sex on pseudo-units
    tmp2 = ad.obs[[unit_key2, sex_key]].copy()
    unit_sex2 = tmp2.groupby(unit_key2)[sex_key].agg(lambda x: x.value_counts().index[0])
    print("[INFO] Pseudo-units per sex:", unit_sex2.value_counts().to_dict())
else:
    unit_key2 = unit_key
    unit_sex2 = unit_sex

# -----------------------------
# Compute sparse unit×celltype mean expression for LR genes
# -----------------------------
# X_lr is aligned to original adata cells; subset it to 'ad'
if not sparse.issparse(X_lr):
    X_lr = sparse.csr_matrix(X_lr)
X_sub = X_lr[mask, :].tocsr()

celltype = ad.obs[celltype_key].astype(str).values
unit_ids = ad.obs[unit_key2].astype(str).values

# encode (unit,celltype) as integer code to avoid string parsing issues
ct_cats = pd.Categorical(celltype)
ct_codes = ct_cats.codes
ct_levels = ct_cats.categories.astype(str).tolist()
n_ct = len(ct_levels)

unit_cats = pd.Categorical(unit_ids)
u_codes = unit_cats.codes
u_levels = unit_cats.categories.astype(str).tolist()
n_u = len(u_levels)

group_code = u_codes.astype(np.int64) * np.int64(n_ct) + ct_codes.astype(np.int64)

means, group_ids, counts = sparse_group_mean(X_sub, group_code, min_cells=MIN_CELLS_PER_GROUP)

# decode groups
u_idx = (group_ids // n_ct).astype(int)
ct_idx = (group_ids % n_ct).astype(int)

group_df = pd.DataFrame({
    "unit": [u_levels[i] for i in u_idx],
    "celltype": [ct_levels[i] for i in ct_idx],
    "n_cells": counts.astype(int),
})

# map unit->sex
group_df["sex"] = group_df["unit"].map(unit_sex2).astype(str)
# drop missing sex units
group_df = group_df[group_df["sex"].isin(["female","male"])].copy()
if group_df.empty:
    raise RuntimeError("After grouping, no unit×celltype groups had sex in {'female','male'}.")

print(f"[INFO] Kept unit×celltype groups: {len(group_df)} | units={group_df['unit'].nunique()} | celltypes={group_df['celltype'].nunique()}")

# -----------------------------
# [5] Compute adjacency per unit
# -----------------------------
print("\n[5] Computing CCC adjacency per unit...")

# Prepare receptor positions as arrays
rec_pos_arr = [np.array(lst, dtype=int) for lst in rec_pos_list]
n_pairs = len(lig_pos)

celltypes = sorted(group_df["celltype"].unique().tolist())
ct_to_i = {c:i for i,c in enumerate(celltypes)}
n_ct_use = len(celltypes)

units = sorted(group_df["unit"].unique().tolist())

# index rows in 'means' correspond to group_df rows (after our filtering we need to re-align)
# We kept means for all groups, but group_df filtered; rebuild a filtered means matrix aligned with group_df:
keep_mask = group_df.index.values
# group_df was built from decoded group_ids in same order as 'means' rows; then filtered by sex -> indices mismatch.
# safest: rebuild from original decoded arrays:
# We'll re-create a boolean selection on the "pre-filter" table:
pre = pd.DataFrame({
    "unit": [u_levels[i] for i in u_idx],
    "celltype": [ct_levels[i] for i in ct_idx],
    "n_cells": counts.astype(int),
})
pre["sex"] = pre["unit"].map(unit_sex2).astype(str)
pre_keep = pre["sex"].isin(["female","male"]).values
means_kept = means[pre_keep, :]
pre_kept = pre[pre_keep].reset_index(drop=True)

# Build per-unit matrices
unit_adj = {}
for u in units:
    sel = (pre_kept["unit"].values == u)
    if sel.sum() == 0:
        continue
    M = means_kept[sel, :]  # (n_celltypes_in_unit x n_genes_used)
    cts = pre_kept.loc[sel, "celltype"].astype(str).values

    # dense small: celltype x genes
    E = np.zeros((n_ct_use, M.shape[1]), dtype=np.float32)
    for r, c in enumerate(cts):
        if c in ct_to_i:
            E[ct_to_i[c], :] = M[r, :]

    # L and R
    L = E[:, lig_pos]  # (ct x pairs)
    R = np.zeros((n_ct_use, n_pairs), dtype=np.float32)
    for j in range(n_pairs):
        R[:, j] = np.min(E[:, rec_pos_arr[j]], axis=1)

    A = (L @ R.T).astype(np.float32)
    unit_adj[u] = A

print(f"[OK] Built adjacency for units: {len(unit_adj)} | celltypes={n_ct_use} | LR_pairs={n_pairs}")

# -----------------------------
# [6] Differential edges
# -----------------------------
print("\n[6] Differential edges (female vs male)...")

unit_list = sorted(unit_adj.keys())
unit_sex_vec = np.array([unit_sex2.loc[u] for u in unit_list], dtype=object)

f_mask = (unit_sex_vec == "female")
m_mask = (unit_sex_vec == "male")
print(f"[INFO] Units for testing: female={f_mask.sum()} male={m_mask.sum()} (pseudo={use_pseudo})")

if f_mask.sum() < 2 or m_mask.sum() < 2:
    print("[WARN] Still <2 units per sex. Skipping p-values; reporting effect sizes only.")
    do_stats = False
else:
    do_stats = True

Xedges = np.stack([unit_adj[u].reshape(-1) for u in unit_list], axis=0)  # (n_units x n_edges)
mean_f = Xedges[f_mask].mean(axis=0)
mean_m = Xedges[m_mask].mean(axis=0)
effect = mean_f - mean_m

# active set
active_thr = np.quantile(np.maximum(mean_f, mean_m), ACTIVE_EDGE_QUANTILE)
active = np.maximum(mean_f, mean_m) >= active_thr

pvals = np.ones(Xedges.shape[1], dtype=float)
qvals = np.ones_like(pvals)

if do_stats:
    _, p_active = stats.ttest_ind(Xedges[f_mask][:, active], Xedges[m_mask][:, active],
                                  axis=0, equal_var=False, nan_policy="omit")
    pvals[active] = np.clip(p_active, 0, 1)
    qvals[active] = storey_qvalues(pvals[active], lam=STOREY_LAMBDA)

# build edge table
edges = [(s, r) for s in celltypes for r in celltypes]
res = pd.DataFrame({
    "sender": [s for (s, r) in edges],
    "receiver": [r for (s, r) in edges],
    "mean_female": mean_f,
    "mean_male": mean_m,
    "effect_f_minus_m": effect,
    "pval": pvals,
    "q_storey": qvals,
    "active": active,
})
res["abs_effect"] = np.abs(res["effect_f_minus_m"])
res.to_csv(RESULTS_DIR / "ccc_edges_by_unit.tsv", sep="\t", index=False)

sig = res[(res["active"]) & (res["q_storey"] <= Q_CUTOFF)].copy() if do_stats else pd.DataFrame()
print(f"[INFO] Active edge threshold (q={ACTIVE_EDGE_QUANTILE:.2f}): {active_thr:.4g}")
if do_stats:
    print(f"[INFO] Exploratory edges (Storey q<= {Q_CUTOFF} on active set): {len(sig)}")

# Print top 5
if do_stats and len(sig) >= TOP_EDGES_TO_PRINT:
    top = sig.sort_values("abs_effect", ascending=False).head(TOP_EDGES_TO_PRINT)
else:
    top = res[res["active"]].sort_values("abs_effect", ascending=False).head(TOP_EDGES_TO_PRINT)

print("\n[TOP EDGES]")
print(top[["sender","receiver","mean_female","mean_male","effect_f_minus_m","pval","q_storey"]].to_string(index=False))

# -----------------------------
# [7] Plots
# -----------------------------
print("\n[7] Plotting...")

A_f = np.mean([unit_adj[u] for u in unit_list if unit_sex2.loc[u] == "female"], axis=0)
A_m = np.mean([unit_adj[u] for u in unit_list if unit_sex2.loc[u] == "male"], axis=0)
A_d = A_f - A_m

A_f_df = pd.DataFrame(A_f, index=celltypes, columns=celltypes)
A_m_df = pd.DataFrame(A_m, index=celltypes, columns=celltypes)
A_d_df = pd.DataFrame(A_d, index=celltypes, columns=celltypes)

vmax = np.quantile(np.r_[A_f.reshape(-1), A_m.reshape(-1)], 0.98)
heatmap(A_f_df, "CCC adjacency (female mean)", "ccc_adj_female", vmax=vmax)
heatmap(A_m_df, "CCC adjacency (male mean)", "ccc_adj_male", vmax=vmax)
heatmap(A_d_df, "CCC adjacency Δ (female - male)", "ccc_adj_delta", vmax=None)

# Volcano (only meaningful if do_stats)
if do_stats:
    dfv = res[res["active"]].copy()
    x = dfv["effect_f_minus_m"].values
    y = -np.log10(np.clip(dfv["pval"].values, 1e-300, 1.0))
    plt.figure(figsize=(8, 6))
    plt.scatter(x, y, s=10, alpha=0.6)
    plt.xlabel("Effect (female - male)")
    plt.ylabel("-log10(p)")
    plt.title("CCC edge differential (active edges)", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "volcano_active_edges.png", dpi=300)
    plt.savefig(RESULTS_DIR / "volcano_active_edges.svg")
    plt.show()

# Barplot top edges (more edges)
if do_stats and len(sig) > 0:
    top_plot = sig.sort_values("abs_effect", ascending=False).head(TOP_EDGES_TO_PLOT).copy()
else:
    top_plot = res[res["active"]].sort_values("abs_effect", ascending=False).head(TOP_EDGES_TO_PLOT).copy()

top_plot["edge"] = top_plot["sender"] + " → " + top_plot["receiver"]

plt.figure(figsize=(10, max(4, 0.30 * len(top_plot))))
sns.barplot(data=top_plot, y="edge", x="effect_f_minus_m")
plt.axvline(0, lw=1)
plt.title(f"Top CCC differential edges (n={len(top_plot)})", fontsize=14, fontweight="bold")
plt.xlabel("Effect (female - male)")
plt.ylabel("")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "top_edges_barplot.png", dpi=300)
plt.savefig(RESULTS_DIR / "top_edges_barplot.svg")
plt.show()

print("\n[DONE] Outputs in:", RESULTS_DIR.resolve())

# -----------------------------
# 8) OPTIONAL: Cell2cell-style plots for each sex (using aggregated celltype means)
#     (This gives you clustermap_ccc, circos, dot_plot, etc.)
# -----------------------------
if HAVE_CELL2CELL:
    print("\n[8] Optional: cell2cell plotting on sex-aggregated celltype means...")

    # Build sex×celltype means (dense small): groups = sex||celltype
    sex_ct = ad.obs[sex_key].astype(str).values + "||" + ad.obs[celltype_key].astype(str).values
    means_sc, groups_sc, _ = _sparse_group_mean(X_lr_sub.tocsr(), sex_ct, min_cells=MIN_CELLS_PER_GROUP)

    meta_sc = pd.DataFrame({"group": groups_sc})
    meta_sc["sex"] = meta_sc["group"].str.split("||", expand=True)[0]
    meta_sc["celltype"] = meta_sc["group"].str.split("||", expand=True)[1]

    # split by sex into gene x "celltype" matrices
    for sx in ["female", "male"]:
        sel = meta_sc["sex"].values == sx
        if sel.sum() == 0:
            continue

        M = means_sc[sel, :]   # (celltypes x genes_used)
        cts = meta_sc.loc[sel, "celltype"].astype(str).values

        # create genes x pseudo-cells dataframe expected by SingleCellInteractions
        # columns are pseudo-barcodes = celltypes
        expr = pd.DataFrame(M, index=cts)  # celltype x genes_used
        expr = expr.T                      # genes_used x celltype

        # metadata: one row per pseudo-cell (celltype)
        meta = pd.DataFrame({"celltype": cts})
        meta["index"] = cts

        # ppi_data = LR pairs mapped (must be str)
        ppi = lr_pairs_mapped.copy().astype(str)

        interactions = c2c.analysis.SingleCellInteractions(
            rnaseq_data=expr,                 # genes x pseudo-cells
            ppi_data=ppi,
            metadata=meta,
            interaction_columns=("ligand_symbol", "receptor_symbol"),
            communication_score="expression_thresholding",
            expression_threshold=0.05,
            cci_score="bray_curtis",
            cci_type="undirected",
            aggregation_method="mean",        # already aggregated
            barcode_col="index",
            celltype_col="celltype",
            complex_sep="&",
            verbose=False
        )

        interactions.compute_pairwise_communication_scores()
        interactions.compute_pairwise_cci_scores()

        # Plot CCC clustermap
        fig = c2c.plotting.clustermap_ccc(
            interactions,
            metric="jaccard",
            method="complete",
            title=f"Active LR pairs for interacting celltypes ({sx})",
            filename=str(RESULTS_DIR / f"cell2cell_ccc_clustermap_{sx}.svg"),
            cell_labels=("SENDER-CELLS", "RECEIVER-CELLS"),
            figsize=(10, 9),
        )
        plt.show()

        # Plot CCI clustermap
        fig = c2c.plotting.clustermap_cci(
            interactions,
            method="complete",
            title=f"CCI scores for celltypes ({sx})",
            filename=str(RESULTS_DIR / f"cell2cell_cci_clustermap_{sx}.svg"),
            cmap="Blues",
        )
        plt.show()

    print("[OK] Saved cell2cell-style figures in:", RESULTS_DIR.resolve())

print("\n[DONE] Outputs written to:", RESULTS_DIR.resolve())


In [ ]:
# =========================================================
# EXTRA CELL2CELL-STYLE VISUALIZATIONS FOR YOUR CCC RESULTS
# =========================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Optional but recommended (provides circos/dotplot/pcoa wrappers)
try:
    import cell2cell as c2c
    HAS_C2C = True
except Exception as e:
    HAS_C2C = False
    print("[WARN] cell2cell not available -> will fall back to seaborn/matplotlib only.")
    print("       Install: pip install cell2cell")

sns.set_style("whitegrid")


# -----------------------------
# 1) Build group metadata (Celltype -> Group)
# -----------------------------
meta_ct = pd.DataFrame({"Celltype": list(celltypes)})

# Heuristic grouping (edit mapping as you like)
def _auto_group(ct: str) -> str:
    c = ct.lower()
    # lymphoid
    if any(x in c for x in ["t", "cd4", "cd8", "nk", "b", "plasma"]):
        return "Lymphoid"
    # myeloid
    if any(x in c for x in ["mono", "mac", "microgl", "dc", "neutro", "mast"]):
        return "Myeloid"
    # stromal/endo/epi
    if any(x in c for x in ["endo", "vasc"]):
        return "Endothelial"
    if any(x in c for x in ["astro", "oligo", "opc", "neur", "glia"]):
        return "Neural/Glial"
    if any(x in c for x in ["epi"]):
        return "Epithelial"
    return "Other"

meta_ct["Group"] = meta_ct["Celltype"].map(_auto_group)

# You can override here manually if needed:
# meta_ct.loc[meta_ct["Celltype"].isin(["pDC","mDC"]), "Group"] = "Myeloid"

print("\n[Group metadata]")
print(meta_ct.head(20))


# -----------------------------
# 2) Colors for groups
# -----------------------------
if HAS_C2C:
    colors = c2c.plotting.get_colors_from_labels(
        labels=meta_ct["Group"].unique().tolist(),
        cmap="tab10"
    )
else:
    # fallback: simple palette
    uniq = meta_ct["Group"].unique().tolist()
    pal = sns.color_palette("tab10", n_colors=len(uniq))
    colors = {g: pal[i] for i, g in enumerate(uniq)}

print("\n[Group colors]")
print(colors)


# -----------------------------
# 3) Helper to order celltypes by group (nice block structure)
# -----------------------------
group_order = meta_ct.sort_values(["Group", "Celltype"])["Celltype"].tolist()

A_f_ord = A_f_df.loc[group_order, group_order]
A_m_ord = A_m_df.loc[group_order, group_order]
A_d_ord = A_d_df.loc[group_order, group_order]


# -----------------------------
# 4) Clustermap-like heatmaps (CCC female/male/delta)
# -----------------------------
def heatmap_blocked(A, title, fname, vmax=None, center=None):
    plt.figure(figsize=(12, 10))
    sns.heatmap(
        A,
        cmap="viridis" if center is None else "coolwarm",
        vmax=vmax,
        center=center,
        square=True,
        cbar_kws={"shrink": 0.75},
    )
    plt.title(title, fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f"{fname}.png", dpi=300)
    plt.savefig(RESULTS_DIR / f"{fname}.svg")
    plt.show()

vmax_common = np.quantile(np.r_[A_f_ord.values.ravel(), A_m_ord.values.ravel()], 0.98)
heatmap_blocked(A_f_ord, "CCC adjacency (female mean) — grouped", "ccc_female_grouped", vmax=vmax_common)
heatmap_blocked(A_m_ord, "CCC adjacency (male mean) — grouped", "ccc_male_grouped", vmax=vmax_common)
heatmap_blocked(A_d_ord, "CCC Δ (female - male) — grouped", "ccc_delta_grouped", center=0.0)


# -----------------------------
# 5) Cell2Cell-style clustermap CCC (requires cell2cell)
#    We emulate an InteractionSpace object using a small adapter.
# -----------------------------
if HAS_C2C:
    # Build a minimal "interaction_space" dataframe like cell2cell expects:
    # rows: "LR pair" and columns: "SENDER;RECEIVER"
    #
    # We don’t have per-LR activity matrix (like full CCC LR-layer).
    # Instead, we convert your aggregated CCC adjacency into a pseudo "interaction_space"
    # with one synthetic "EDGE_SCORE" row.
    #
    # This is purely for visualization layout (cluster + metadata colors).
    cols = [f"{s};{r}" for s in group_order for r in group_order]
    pseudo = pd.DataFrame([A_f_ord.values.reshape(-1)], index=["EDGE_SCORE"], columns=cols)

    # Cell2Cell clustermap_ccc expects an InteractionSpace-like object,
    # but it also works on a DataFrame-like in many setups; if not, fallback to seaborn.
    try:
        interaction_clustermap = c2c.plotting.clustermap_ccc(
            pseudo,
            metric="jaccard",
            method="complete",
            metadata=meta_ct,
            sample_col="Celltype",
            group_col="Group",
            colors=colors,
            row_fontsize=12,
            title="CCC (female) — grouped (Cell2Cell-style layout)",
            filename=str(RESULTS_DIR / "cell2cellstyle_ccc_female_clustermap.svg"),
            cell_labels=("SENDER-CELLS", "RECEIVER-CELLS"),
            **{"figsize": (11, 9)},
        )
        # Legend
        c2c.plotting.generate_legend(
            color_dict=colors,
            loc="center left",
            bbox_to_anchor=(1.2, 0.5),
            ncol=1,
            fancybox=True,
            shadow=True,
            title="Groups",
            fontsize=12,
        )
        plt.show()
    except Exception as e:
        print("[WARN] cell2cell clustermap_ccc failed on pseudo matrix. Using seaborn heatmap only.", repr(e))


# -----------------------------
# 6) Circos plot on TOP edges (from your res table)
# -----------------------------
# Choose a set of "interesting" edges to visualize:
# - if q-values exist, use q<=0.25 and top abs_effect
# - else just top abs_effect among active edges
res_use = res.copy()
if "q_storey" in res_use.columns:
    cand = res_use[(res_use["active"]) & (res_use["q_storey"] <= 0.25)].copy()
    if len(cand) < 10:
        cand = res_use[res_use["active"]].copy()
else:
    cand = res_use[res_use["active"]].copy()

cand = cand.sort_values("abs_effect", ascending=False).head(40)

# pick top senders/receivers from those edges
top_senders = cand["sender"].value_counts().head(6).index.tolist()
top_receivers = cand["receiver"].value_counts().head(6).index.tolist()

# build a circos-friendly list: sender_cells, receiver_cells
sender_cells = top_senders
receiver_cells = top_receivers

print("\n[Circos] sender_cells:", sender_cells)
print("[Circos] receiver_cells:", receiver_cells)

if HAS_C2C:
    # Build an InteractionSpace-like object for circos:
    # cell2cell circos_plot wants a "interaction_space" with LR-level scores.
    # We approximate with edge scores (sender->receiver) as one row, same as above.
    cols = [f"{s};{r}" for s in sender_cells for r in receiver_cells]
    edge_mat = A_d_df.loc[sender_cells, receiver_cells].values  # delta
    pseudo2 = pd.DataFrame([edge_mat.reshape(-1)], index=["DELTA_EDGE_SCORE"], columns=cols)

    # Here ligands/receptors are not available (we’re not in LR-pair space).
    # So we use a "circos_plot" variant by passing only interaction_space,
    # and interpret values as edge weights.
    try:
        ax = c2c.plotting.circos_plot(
            interaction_space=pseudo2,
            sender_cells=sender_cells,
            receiver_cells=receiver_cells,
            ligands=[],            # not used here
            receptors=[],          # not used here
            excluded_score=0,
            metadata=meta_ct,
            sample_col="Celltype",
            group_col="Group",
            colors=colors,
            fontsize=12,
        )
        plt.title("CCC Δ (female-male) — circos (edge-level approximation)", fontsize=13, fontweight="bold")
        plt.tight_layout()
        plt.savefig(RESULTS_DIR / "circos_delta_edges.png", dpi=300)
        plt.savefig(RESULTS_DIR / "circos_delta_edges.svg")
        plt.show()
    except Exception as e:
        print("[WARN] cell2cell circos_plot failed on edge-level approximation.", repr(e))
else:
    print("[INFO] Skipping circos_plot (cell2cell not installed).")


# -----------------------------
# 7) Dot plot-like view for top edges (effect + q)
# -----------------------------
topN = 50
dfdot = cand.head(topN).copy()
dfdot["edge"] = dfdot["sender"] + " → " + dfdot["receiver"]

plt.figure(figsize=(10, max(5, 0.22 * len(dfdot))))
x = dfdot["effect_f_minus_m"]
y = np.arange(len(dfdot))[::-1]
sizes = 40 + 200 * (dfdot["abs_effect"] / (dfdot["abs_effect"].max() + 1e-9))

plt.scatter(x, y, s=sizes, alpha=0.7)
plt.axvline(0, lw=1)
plt.yticks(y, dfdot["edge"])
plt.xlabel("Effect (female - male)")
if "q_storey" in dfdot.columns:
    plt.title(f"Top {topN} CCC differential edges (size=|effect|)", fontsize=14, fontweight="bold")
else:
    plt.title(f"Top {topN} CCC edges (size=|effect|)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "dotplot_top_edges.png", dpi=300)
plt.savefig(RESULTS_DIR / "dotplot_top_edges.svg")
plt.show()


# -----------------------------
# 8) CCI-like visualization + PCoA (undirected)
#    Use symmetrized mean adjacency as an "interaction potential"
# -----------------------------
A_und_f = 0.5 * (A_f_df.values + A_f_df.values.T)
A_und_m = 0.5 * (A_m_df.values + A_m_df.values.T)

def pcoa_from_similarity(S, n_components=3):
    # PCoA from a similarity/affinity matrix -> convert to distances
    # distance_ij = sqrt(max(S_ii + S_jj - 2S_ij, 0))
    diag = np.diag(S)
    D2 = diag[:, None] + diag[None, :] - 2 * S
    D2 = np.maximum(D2, 0.0)
    D = np.sqrt(D2 + 1e-12)

    # Classical MDS / PCoA
    n = D.shape[0]
    H = np.eye(n) - np.ones((n, n)) / n
    B = -0.5 * H @ (D2) @ H
    w, V = np.linalg.eigh(B)
    idx = np.argsort(w)[::-1]
    w = w[idx]
    V = V[:, idx]
    wpos = np.maximum(w[:n_components], 0)
    coords = V[:, :n_components] * np.sqrt(wpos + 1e-12)
    return coords

coords_f = pcoa_from_similarity(A_und_f, n_components=3)
coords_m = pcoa_from_similarity(A_und_m, n_components=3)

# Plot PCoA 2D
def plot_pcoa(coords, title, fname):
    dfp = pd.DataFrame(coords[:, :2], columns=["PC1", "PC2"])
    dfp["Celltype"] = celltypes
    dfp = dfp.merge(meta_ct, on="Celltype", how="left")

    plt.figure(figsize=(8, 6))
    for g in dfp["Group"].unique():
        sub = dfp[dfp["Group"] == g]
        plt.scatter(sub["PC1"], sub["PC2"], label=g, s=60, alpha=0.8)
        for _, row in sub.iterrows():
            plt.text(row["PC1"], row["PC2"], row["Celltype"], fontsize=8)

    plt.title(title, fontsize=14, fontweight="bold")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f"{fname}.png", dpi=300)
    plt.savefig(RESULTS_DIR / f"{fname}.svg")
    plt.show()

plot_pcoa(coords_f, "PCoA (female) — undirected CCI-like", "pcoa_female_cci_like")
plot_pcoa(coords_m, "PCoA (male) — undirected CCI-like", "pcoa_male_cci_like")

print("\n[OK] Extra visualizations saved in:", RESULTS_DIR.resolve())


In [ ]:
# ============================================================
# Cell2Cell-style CCC visualizations (clustermap, circos, dots)
# applied to YOUR dataset with FEMALE vs MALE (mouse symbols)
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import cell2cell as c2c
import scanpy as sc
from scipy import sparse

# -------------------------
# CONFIG (tune if needed)
# -------------------------
MIN_CELLS_PER_DONOR_CELLTYPE = 25
MAX_LR_PAIRS_FOR_PLOTTING = 800          # keeps clustermaps fast
N_PERMUTATIONS = 50                      # for dot_plot significance; increase later (100-1000)
EXPR_THRESHOLD = 0.10                    # like tutorial (after aggregation)
SIGNIF_FOR_DOTPLOT = 0.20                # tutorial uses 0.2
CIRCO_N_LIGANDS = 3
CIRCO_N_RECEPTORS = 3
CIRCO_N_SENDERS = 3
CIRCO_N_RECEIVERS = 5

# Which expression to use from AnnData
# (cell2cell expects non-negative aggregated expression; log1p is fine with thresholding)
LAYER_USE = None   # set to "lognorm" if you store it; else None -> adata.X

# Gene symbol columns to try (mouse dataset)
GENE_SYMBOL_CANDIDATES = ["gene_symbols", "feature_name", "gene_symbol", "symbol"]

# -------------------------
# 0) Utilities
# -------------------------
def pick_var_key(adata, candidates):
    for k in candidates:
        if k in adata.var.columns:
            return k
    return None

def _csr(X):
    return X.tocsr() if sparse.issparse(X) else sparse.csr_matrix(np.asarray(X))

def pseudobulk_mean_by_group(ad, group_key, layer=None):
    """Return mean expression per group (groups x genes) + n_cells per group."""
    X = ad.layers[layer] if (layer is not None and layer in ad.layers) else ad.X
    X = _csr(X)
    groups = ad.obs[group_key].astype(str).values
    cats, inv = np.unique(groups, return_inverse=True)

    n_groups = len(cats)
    n_cells = ad.n_obs
    G = sparse.csr_matrix(
        (np.ones(n_cells, dtype=np.float32), (inv, np.arange(n_cells))),
        shape=(n_groups, n_cells),
    )
    sums = G @ X  # (groups x genes)
    counts = np.bincount(inv, minlength=n_groups).astype(np.float32)
    counts[counts == 0] = 1.0
    means = sums.multiply(1.0 / counts[:, None]) if sparse.issparse(sums) else (sums / counts[:, None])
    means = means.toarray() if sparse.issparse(means) else np.asarray(means)

    mean_df = pd.DataFrame(means, index=cats, columns=ad.var_names)
    n_df = pd.Series(counts.astype(int), index=cats, name="n_cells")
    return mean_df, n_df

def split_complex(s, sep="&"):
    return [x.strip() for x in str(s).split(sep) if x.strip()]

# -------------------------
# 1) Ensure gene symbols as var_names (mouse)
# -------------------------
gene_sym_key = pick_var_key(adata, GENE_SYMBOL_CANDIDATES)
if gene_sym_key is None:
    raise KeyError(f"Could not find gene symbol column in adata.var among: {GENE_SYMBOL_CANDIDATES}")

# Build a working view where var_names are gene symbols (unique)
ad_sym = adata.copy()
ad_sym.var_names = ad_sym.var[gene_sym_key].astype(str).values
ad_sym.var_names_make_unique()

print("[INFO] Using gene symbols from:", f"adata.var['{gene_sym_key}']")

# -------------------------
# 2) Load or reuse LR pairs (mouse)
# -------------------------
if "lr_pairs_mapped" in globals() and isinstance(lr_pairs_mapped, pd.DataFrame):
    lr_df = lr_pairs_mapped.copy()
    # ensure expected columns
    if not set(["ligand_symbol", "receptor_symbol"]).issubset(lr_df.columns):
        raise KeyError("lr_pairs_mapped must contain columns: ligand_symbol, receptor_symbol")
    print("[OK] Using existing lr_pairs_mapped | rows=", len(lr_df))
else:
    url = "https://raw.githubusercontent.com/LewisLabUCSD/Ligand-Receptor-Pairs/master/Mouse/Mouse-2020-Jin-LR-pairs.csv"
    lr_df = pd.read_csv(url)
    # robust column selection
    # many of these LR files have ligand_symbol/receptor_symbol already
    if "ligand_symbol" not in lr_df.columns or "receptor_symbol" not in lr_df.columns:
        # try common alternatives
        cand_l = [c for c in lr_df.columns if "ligand" in c.lower() and "symbol" in c.lower()]
        cand_r = [c for c in lr_df.columns if "receptor" in c.lower() and "symbol" in c.lower()]
        if len(cand_l) == 0 or len(cand_r) == 0:
            raise KeyError("Could not find ligand_symbol/receptor_symbol in the LR file columns.")
        lr_df = lr_df.rename(columns={cand_l[0]: "ligand_symbol", cand_r[0]: "receptor_symbol"})
    lr_df = lr_df[["ligand_symbol", "receptor_symbol"]].astype(str)
    print("[OK] Downloaded mouse LR pairs | rows=", len(lr_df))

# Build LR gene universe (expand complexes)
lr_genes = set()
for _, row in lr_df.iterrows():
    for g in split_complex(row["ligand_symbol"], "&"):
        lr_genes.add(g)
    for g in split_complex(row["receptor_symbol"], "&"):
        lr_genes.add(g)

# Overlap check
overlap = len(set(ad_sym.var_names) & lr_genes)
print("[INFO] LR genes:", len(lr_genes), "| overlap with dataset symbols:", overlap)
if overlap < 200:
    raise RuntimeError(
        f"Too few LR genes overlap ({overlap}). "
        f"Check gene symbols (mouse) and that adata.var['{gene_sym_key}'] is correct."
    )

# Subset dataset to LR genes to speed up everything
keep_genes = [g for g in ad_sym.var_names if g in lr_genes]
ad_lr = ad_sym[:, keep_genes].copy()
print("[INFO] Subset to LR genes:", ad_lr.shape)

# -------------------------
# 3) Create donor×celltype pseudo-cells per sex (small & fast)
# -------------------------
def build_pseudocells_for_sex(ad, sex_value):
    adx = ad[ad.obs[sex_key].astype(str) == sex_value].copy()
    if adx.n_obs == 0:
        return None, None

    # group id = donor__celltype
    adx.obs["_donor"] = adx.obs[sample_key].astype(str)
    adx.obs["_ct"] = adx.obs[celltype_key].astype(str)
    adx.obs["_group"] = adx.obs["_donor"] + "__" + adx.obs["_ct"]

    # compute mean per donor×celltype
    mean_df, n_cells = pseudobulk_mean_by_group(adx, "_group", layer=LAYER_USE)

    # filter small groups
    keep_groups = n_cells[n_cells >= MIN_CELLS_PER_DONOR_CELLTYPE].index.tolist()
    mean_df = mean_df.loc[keep_groups]
    n_cells = n_cells.loc[keep_groups]

    # meta for pseudo-cells
    meta = pd.DataFrame({"index": mean_df.index})
    meta["donor"] = meta["index"].str.split("__").str[0]
    meta["celltype"] = meta["index"].str.split("__").str[1]
    meta["sex"] = sex_value
    meta["n_cells"] = n_cells.values

    # rnaseq_data expected by cell2cell is genes x pseudo-cells
    rnaseq_data = mean_df.T  # genes x pseudo-cells
    return rnaseq_data, meta

rnaseq_f, meta_f = build_pseudocells_for_sex(ad_lr, female_label)
rnaseq_m, meta_m = build_pseudocells_for_sex(ad_lr, male_label)

print("\n[INFO] Pseudocells:")
if meta_f is not None:
    print("  female pseudo-cells:", rnaseq_f.shape[1], "| donors:", meta_f["donor"].nunique(), "| celltypes:", meta_f["celltype"].nunique())
if meta_m is not None:
    print("  male   pseudo-cells:", rnaseq_m.shape[1], "| donors:", meta_m["donor"].nunique(), "| celltypes:", meta_m["celltype"].nunique())

if meta_f is None or meta_m is None:
    raise RuntimeError("Missing pseudo-cells for one sex. Check sex labels and filtering thresholds.")

# -------------------------
# 4) Filter LR pairs for plotting (top by expression evidence)
# -------------------------
# Score LR pairs using global mean expression in pseudo-cells (female+male combined)
rnaseq_all = pd.concat([rnaseq_f, rnaseq_m], axis=1)
gene_mean = rnaseq_all.mean(axis=1)

def lr_pair_score(lig, rec):
    ligs = split_complex(lig, "&")
    recs = split_complex(rec, "&")
    # for complexes: use min across subunits (conservative)
    lig_score = min([gene_mean.get(g, 0.0) for g in ligs]) if len(ligs) else 0.0
    rec_score = min([gene_mean.get(g, 0.0) for g in recs]) if len(recs) else 0.0
    return float(lig_score * rec_score)

lr_df2 = lr_df.copy()
lr_df2["score"] = [lr_pair_score(l, r) for l, r in zip(lr_df2["ligand_symbol"], lr_df2["receptor_symbol"])]
lr_df2 = lr_df2.sort_values("score", ascending=False).head(MAX_LR_PAIRS_FOR_PLOTTING).drop(columns=["score"]).reset_index(drop=True)
print(f"[INFO] Using LR pairs for plotting: {len(lr_df2)} (top by expression evidence)")

# -------------------------
# 5) Build Cell2Cell SingleCellInteractions objects (female & male)  [FIXED]
# -------------------------
def build_interactions(rnaseq_data, meta, label):
    """
    rnaseq_data: genes x pseudo-cells (DataFrame)
    meta: DataFrame with columns ['index', 'celltype', ...] where 'index' matches rnaseq_data columns
    """
    # Cell2Cell expects 'aggregation_method' in {'average','nn_cell_fraction'}
    # Even if pseudo-cells are already aggregated, use 'average' (safe & supported).
    aggregation_method = "average"

    # Make sure the barcode column matches rnaseq_data columns
    meta = meta.copy()
    meta["index"] = meta["index"].astype(str)
    rnaseq_data = rnaseq_data.copy()
    rnaseq_data.columns = rnaseq_data.columns.astype(str)

    # Subset meta to existing columns in rnaseq_data (defensive)
    meta = meta[meta["index"].isin(rnaseq_data.columns)].copy()

    interactions = c2c.analysis.SingleCellInteractions(
        rnaseq_data=rnaseq_data,              # genes x pseudo-cells
        ppi_data=lr_df2,
        metadata=meta,
        interaction_columns=("ligand_symbol", "receptor_symbol"),
        communication_score="expression_thresholding",
        expression_threshold=EXPR_THRESHOLD,  # threshold after aggregation
        cci_score="bray_curtis",
        cci_type="undirected",
        aggregation_method=aggregation_method,  # <-- FIX
        barcode_col="index",
        celltype_col="celltype",
        complex_sep="&",
        verbose=False
    )

    print(f"\n[{label}] compute_pairwise_communication_scores() ...")
    interactions.compute_pairwise_communication_scores()
    print(f"[{label}] compute_pairwise_cci_scores() ...")
    interactions.compute_pairwise_cci_scores()

    # permutations for dot_plot significance
    try:
        print(f"[{label}] permute_cell_labels (N={N_PERMUTATIONS}) ...")
        interactions.permute_cell_labels(
            evaluation="communication",
            permutations=N_PERMUTATIONS,
            fdr_correction=True,
            verbose=False
        )
        interactions.permute_cell_labels(
            evaluation="interactions",
            permutations=N_PERMUTATIONS,
            fdr_correction=True,
            verbose=False
        )
    except Exception as e:
        print(f"[WARN] permutations failed for {label}: {repr(e)}")
        print("       dot_plot may still work but without significance filtering.")
    return interactions

# Re-run
interactions_f = build_interactions(rnaseq_f, meta_f, "FEMALE")
interactions_m = build_interactions(rnaseq_m, meta_m, "MALE")

# -------------------------
# 6) Build group_meta + colors (Celltype -> Group)
# -------------------------
# union of celltypes across sexes
all_celltypes = sorted(list(set(meta_f["celltype"].unique()) | set(meta_m["celltype"].unique())))
group_meta = pd.DataFrame({"Celltype": all_celltypes})

# Auto-grouping (edit if you want a hand-crafted mapping like the tutorial example)
def auto_group(ct):
    c = ct.lower()
    if any(x in c for x in ["cd4", "cd8", "t cell", "tcell", "nk", "b cell", "bcell", "plasma"]):
        return "Lymphoid"
    if any(x in c for x in ["mono", "mac", "microgl", "dc", "neutro", "mast"]):
        return "Myeloid"
    if any(x in c for x in ["endo", "vasc"]):
        return "Endothelial"
    if any(x in c for x in ["astro", "oligo", "opc", "neur", "glia"]):
        return "Neural/Glial"
    if "epi" in c:
        return "Epithelial"
    return "Other"

group_meta["Group"] = group_meta["Celltype"].map(auto_group)

colors = c2c.plotting.get_colors_from_labels(
    labels=group_meta["Group"].unique().tolist(),
    cmap="tab10"
)
print("\n[INFO] group_meta head:\n", group_meta.head())

# -------------------------
# 7) Plot: clustermap_ccc (female + male)
# -------------------------
def plot_clustermap_ccc(interactions, label):
    fn = str(RESULTS_DIR / f"cell2cell_clustermap_ccc_{label}.svg")
    cm = c2c.plotting.clustermap_ccc(
        interactions,
        metric="jaccard",
        method="complete",
        metadata=group_meta,
        sample_col="Celltype",
        group_col="Group",
        colors=colors,
        row_fontsize=10,
        title=f"Active ligand-receptor pairs for interacting cells ({label})",
        filename=fn,
        cell_labels=("SENDER-CELLS", "RECEIVER-CELLS"),
        **{"figsize": (10, 9)}
    )
    c2c.plotting.generate_legend(
        color_dict=colors,
        loc="center left",
        bbox_to_anchor=(1.15, 0.5),
        ncol=1,
        fancybox=True,
        shadow=True,
        title="Groups",
        fontsize=11,
    )
    plt.show()

plot_clustermap_ccc(interactions_f, "female")
plot_clustermap_ccc(interactions_m, "male")

# -------------------------
# 8) Plot: circos_plot (robust, non-crashing)
# -------------------------
import re
import difflib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cell2cell as c2c

def _unwrap_space(obj):
    """Accept SingleCellInteractions or InteractionSpace."""
    if hasattr(obj, "interaction_space"):
        inner = getattr(obj, "interaction_space")
        if inner is not None and not isinstance(inner, (pd.DataFrame, pd.Series, np.ndarray)):
            return inner
    return obj

def _norm(s: str) -> str:
    return re.sub(r"\s+", " ", str(s).strip().lower())

def _fuzzy_map(requested, available, cutoff=0.60):
    """
    Map requested names onto available names using normalization + fuzzy matching.
    Returns (mapped_list, report_dict).
    """
    if requested is None:
        return None, {"mapped": None, "unmatched": [], "suggestions": {}}

    available = [str(x) for x in available]
    avail_norm = {_norm(x): x for x in available}

    mapped = []
    unmatched = []
    suggestions = {}

    for r in requested:
        r0 = str(r)
        rn = _norm(r0)
        if rn in avail_norm:
            mapped.append(avail_norm[rn])
            continue
        keys = list(avail_norm.keys())
        close = difflib.get_close_matches(rn, keys, n=3, cutoff=cutoff)
        if close:
            mapped.append(avail_norm[close[0]])
            suggestions[r0] = [avail_norm[c] for c in close]
        else:
            unmatched.append(r0)
            suggestions[r0] = []

    # unique preserve order
    out = []
    seen = set()
    for x in mapped:
        if x not in seen:
            out.append(x); seen.add(x)

    return out, {"mapped": out, "unmatched": unmatched, "suggestions": suggestions}

def _expand_to_complexes(requested, available, sep="&"):
    """
    If user passes gene 'Cd74', expand to any complex strings containing it: 'Cd74&X', etc.
    Also matches case-insensitively.
    """
    if requested is None:
        return None
    requested = [str(x) for x in requested if x is not None]
    if len(requested) == 0:
        return []

    avail = [str(a) for a in pd.unique(pd.Series(list(available)).dropna())]
    avail_lower = {a.lower(): a for a in avail}

    out = []
    for g in requested:
        gl = g.lower()
        if gl in avail_lower:
            out.append(avail_lower[gl])
            continue
        # expand to complexes that contain gl as a subunit
        hits = []
        for a in avail:
            parts = [p.strip().lower() for p in a.split(sep)]
            if gl in parts:
                hits.append(a)
        out.extend(hits)

    # unique preserve order
    uniq = []
    seen = set()
    for x in out:
        if x not in seen:
            uniq.append(x); seen.add(x)
    return uniq

def _standardize_links_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Force a long links DF with columns: sender, receiver, ligand, receptor, score
    using common synonyms.
    """
    df = df.copy()
    colmap = {}
    for c in df.columns:
        lc = str(c).lower()
        if lc in ["sender", "source", "from", "cell_sender", "sending_cell", "celltype_a", "a"]:
            colmap[c] = "sender"
        elif lc in ["receiver", "target", "to", "cell_receiver", "receiving_cell", "celltype_b", "b"]:
            colmap[c] = "receiver"
        elif lc in ["ligand", "ligand_symbol", "ligand_gene", "ligandname"]:
            colmap[c] = "ligand"
        elif lc in ["receptor", "receptor_symbol", "receptor_gene", "receptorname"]:
            colmap[c] = "receptor"
        elif lc in ["score", "communication", "communication_score", "weight", "value", "cci_score"]:
            colmap[c] = "score"

    df = df.rename(columns=colmap)

    need = {"sender", "receiver", "ligand", "receptor"}
    missing = need - set(df.columns)
    if missing:
        raise RuntimeError(
            f"Could not standardize links table. Missing columns {missing}. "
            f"Available columns: {list(df.columns)}"
        )

    if "score" not in df.columns:
        df["score"] = 1.0  # fallback if score not provided

    df["score"] = pd.to_numeric(df["score"], errors="coerce").fillna(0.0)
    for c in ["sender", "receiver", "ligand", "receptor"]:
        df[c] = df[c].astype(str)

    return df[["sender", "receiver", "ligand", "receptor", "score"]]

def _find_links_df(interaction_space):
    """
    Extract a long-format links DF from InteractionSpace (or wrapper).
    """
    space = _unwrap_space(interaction_space)

    # Preferred: InteractionSpace.interaction_elements
    if hasattr(space, "interaction_elements"):
        ie = getattr(space, "interaction_elements")
        if isinstance(ie, pd.DataFrame):
            return ie

    # Otherwise scan attributes for a DF containing ligand/receptor + sender/receiver
    for attr in dir(space):
        if attr.startswith("_"):
            continue
        try:
            v = getattr(space, attr)
        except Exception:
            continue
        if isinstance(v, pd.DataFrame):
            cols = set([str(c).lower() for c in v.columns])
            if any("ligand" in c for c in cols) and any("receptor" in c for c in cols) and (
                any(c in cols for c in ["sender", "source", "from", "celltype_a", "a"]) and
                any(c in cols for c in ["receiver", "target", "to", "celltype_b", "b"])
            ):
                return v

    raise RuntimeError(
        "Could not locate a links DataFrame inside the InteractionSpace. "
        "Try printing dir(interactions_f.interaction_space) and inspect candidate tables."
    )

def plot_circos_safe(
    interaction_space,
    label: str,
    group_meta: pd.DataFrame,
    colors: dict,
    sender_cells=None,
    receiver_cells=None,
    ligands=None,
    receptors=None,
    score_floor: float = 0.0,
    top_edges: int | None = 1500,
    max_senders: int = 6,
    max_receivers: int = 6,
    max_lr: int = 12,
    complex_sep: str = "&",
    relax_if_empty: bool = True,
    verbose: bool = True,
):
    space = _unwrap_space(interaction_space)
    links_raw = _find_links_df(space)
    links = _standardize_links_df(links_raw)

    # score filter
    base0 = links[links["score"] > float(score_floor)].copy()
    if base0.empty and score_floor > 0:
        if verbose:
            print(f"[{label}] No edges above score_floor={score_floor}. Falling back to score_floor=0.")
        base0 = links[links["score"] > 0].copy()

    if base0.empty:
        raise RuntimeError(f"[{label}] No non-zero edges found at all (scores are 0).")

    # keep only top edges for speed/clarity
    if top_edges is not None and len(base0) > int(top_edges):
        base0 = base0.sort_values("score", ascending=False).head(int(top_edges)).copy()

    # map sender/receiver labels to available ones
    send_avail = sorted(base0["sender"].unique())
    recv_avail = sorted(base0["receiver"].unique())
    sender_mapped, rep_s = _fuzzy_map(sender_cells, send_avail)
    receiver_mapped, rep_r = _fuzzy_map(receiver_cells, recv_avail)

    if verbose and sender_cells is not None:
        print(f"[{label}] sender requested={sender_cells}")
        print(f"[{label}] sender mapped={sender_mapped}")
        if rep_s["unmatched"]:
            print(f"[{label}] sender unmatched={rep_s['unmatched']} suggestions={rep_s['suggestions']}")
    if verbose and receiver_cells is not None:
        print(f"[{label}] receiver requested={receiver_cells}")
        print(f"[{label}] receiver mapped={receiver_mapped}")
        if rep_r["unmatched"]:
            print(f"[{label}] receiver unmatched={rep_r['unmatched']} suggestions={rep_r['suggestions']}")

    # apply sender/receiver constraints
    base = base0.copy()
    if sender_mapped:
        base = base[base["sender"].isin(sender_mapped)]
    if receiver_mapped:
        base = base[base["receiver"].isin(receiver_mapped)]

    # If empty: AUTO-RELAX instead of raising
    if base.empty:
        if not relax_if_empty:
            raise RuntimeError(
                f"[{label}] No interactions after sender/receiver constraints. "
                f"Verify your celltype strings match what's in meta_f/meta_m."
            )
        if verbose:
            print(f"[{label}] Empty after sender/receiver constraints -> auto-selecting strong senders/receivers from data.")

        top_s = (base0.groupby("sender")["score"].sum().sort_values(ascending=False).head(int(max_senders)).index.tolist())
        top_r = (base0.groupby("receiver")["score"].sum().sort_values(ascending=False).head(int(max_receivers)).index.tolist())
        base = base0[base0["sender"].isin(top_s) & base0["receiver"].isin(top_r)].copy()

        # still empty? (rare) fall back to base0
        if base.empty:
            base = base0.copy()

    # ligand/receptor expansion + filtering
    lig_avail = base["ligand"].unique()
    rec_avail = base["receptor"].unique()
    lig_exp = _expand_to_complexes(ligands, lig_avail, sep=complex_sep)
    rec_exp = _expand_to_complexes(receptors, rec_avail, sep=complex_sep)

    base_lr = base.copy()
    if lig_exp is not None and len(lig_exp) > 0:
        base_lr = base_lr[base_lr["ligand"].isin(lig_exp)]
    if rec_exp is not None and len(rec_exp) > 0:
        base_lr = base_lr[base_lr["receptor"].isin(rec_exp)]

    # If empty after LR constraints: auto-select top LR pairs
    if base_lr.empty:
        if not relax_if_empty:
            raise RuntimeError(f"[{label}] No edges after ligand/receptor constraints.")
        if verbose:
            print(f"[{label}] Empty after ligand/receptor constraints -> auto-selecting top LR pairs.")
        top_lr = (
            base.assign(lr=base["ligand"].astype(str) + "→" + base["receptor"].astype(str))
                .groupby("lr")["score"].sum()
                .sort_values(ascending=False)
                .head(int(max_lr))
                .index.tolist()
        )
        lr_pairs = [x.split("→", 1) for x in top_lr]
        lig_use = [a for a, b in lr_pairs]
        rec_use = [b for a, b in lr_pairs]
        base_lr = base[base["ligand"].isin(lig_use) & base["receptor"].isin(rec_use)].copy()
    else:
        # Trim to max_lr by evidence
        top_lr = (
            base_lr.assign(lr=base_lr["ligand"].astype(str) + "→" + base_lr["receptor"].astype(str))
                  .groupby("lr")["score"].sum()
                  .sort_values(ascending=False)
                  .head(int(max_lr))
                  .index.tolist()
        )
        lr_pairs = [x.split("→", 1) for x in top_lr]
        lig_use = [a for a, b in lr_pairs]
        rec_use = [b for a, b in lr_pairs]
        base_lr = base_lr[base_lr["ligand"].isin(lig_use) & base_lr["receptor"].isin(rec_use)].copy()

    # final sender/receiver sets based on base_lr
    sender_use = (base_lr.groupby("sender")["score"].sum().sort_values(ascending=False).head(int(max_senders)).index.tolist())
    receiver_use = (base_lr.groupby("receiver")["score"].sum().sort_values(ascending=False).head(int(max_receivers)).index.tolist())

    # ensure group_meta covers used cells
    gm = group_meta.copy()
    if "Celltype" not in gm.columns or "Group" not in gm.columns:
        raise RuntimeError("group_meta must have columns: ['Celltype','Group']")
    used_cells = sorted(set(sender_use) | set(receiver_use))
    missing = [c for c in used_cells if c not in set(gm["Celltype"].astype(str))]
    if missing:
        gm = pd.concat([gm, pd.DataFrame({"Celltype": missing, "Group": ["Other"] * len(missing)})], ignore_index=True)
        if "Other" not in colors:
            colors = {**colors, **c2c.plotting.get_colors_from_labels(labels=["Other"], cmap="tab10")}

    if verbose:
        print(f"[{label}] circos senders={sender_use}")
        print(f"[{label}] circos receivers={receiver_use}")
        print(f"[{label}] circos #LR={len(pd.unique(base_lr['ligand'].astype(str) + '→' + base_lr['receptor'].astype(str)))}")
        print(f"[{label}] edges in plot subset: {len(base_lr)}")

    ax = c2c.plotting.circos_plot(
        interaction_space=space,
        sender_cells=sender_use,
        receiver_cells=receiver_use,
        ligands=sorted(base_lr["ligand"].unique().tolist()),
        receptors=sorted(base_lr["receptor"].unique().tolist()),
        excluded_score=float(score_floor),
        metadata=gm,
        sample_col="Celltype",
        group_col="Group",
        colors=colors,
        fontsize=14,
    )
    plt.title(f"Circos ({label})", fontsize=14)
    return ax

def _find_links_df(interaction_space):
    """
    Try to extract a long-format links df from InteractionSpace.
    1) Prefer .interaction_elements if present.
    2) Otherwise search for any df that looks like long links.
    3) Otherwise search for any df that looks like LR x cellpair matrix and convert.
    """
    space = _unwrap_interaction_space(interaction_space)

    # 1) interaction_elements is the canonical place in InteractionSpace (per stack traces / internals)
    if hasattr(space, "interaction_elements"):
        ie = getattr(space, "interaction_elements")
        if isinstance(ie, pd.DataFrame):
            lig_col, rec_col, send_col, recv_col, score_col = _infer_links_columns(ie)
            if lig_col and rec_col and send_col and recv_col:
                out = ie.copy()
                rename = {lig_col: "ligand", rec_col: "receptor", send_col: "sender", recv_col: "receiver"}
                out = out.rename(columns=rename)
                if score_col and score_col != "score":
                    out = out.rename(columns={score_col: "score"})
                if "score" not in out.columns:
                    # sometimes there is no explicit score column; try common ones
                    for c in ["communication_score", "cci_score", "ppi_score"]:
                        if c in out.columns:
                            out = out.rename(columns={c: "score"})
                            break
                if "score" not in out.columns:
                    out["score"] = np.nan
                return out[["ligand", "receptor", "sender", "receiver", "score"]]

    # 2) search attributes for a long-format df
    for attr in dir(space):
        if attr.startswith("_"):
            continue
        try:
            v = getattr(space, attr)
        except Exception:
            continue
        if isinstance(v, pd.DataFrame):
            lig_col, rec_col, send_col, recv_col, score_col = _infer_links_columns(v)
            if lig_col and rec_col and send_col and recv_col:
                out = v.copy()
                out = out.rename(columns={lig_col: "ligand", rec_col: "receptor", send_col: "sender", recv_col: "receiver"})
                if score_col and score_col != "score":
                    out = out.rename(columns={score_col: "score"})
                if "score" not in out.columns:
                    out["score"] = np.nan
                return out[["ligand", "receptor", "sender", "receiver", "score"]]

    # 3) search for LR x cellpair matrix-like df and convert
    candidates = []
    for attr in dir(space):
        if attr.startswith("_"):
            continue
        try:
            v = getattr(space, attr)
        except Exception:
            continue
        if isinstance(v, pd.DataFrame) and v.shape[0] > 10 and v.shape[1] > 2:
            candidates.append((attr, v))

    # heuristic: choose the biggest numeric-ish df
    best = None
    best_score = -1
    for name, df in candidates:
        num = df.select_dtypes(include=[np.number]).shape[1]
        s = (np.log1p(df.shape[0] * df.shape[1]) +
             0.5 * num +
             (2.0 if isinstance(df.index, pd.MultiIndex) else 0.0) +
             (2.0 if isinstance(df.columns, pd.MultiIndex) else 0.0))
        if s > best_score:
            best_score = s
            best = df

    if best is not None:
        return _matrix_to_links(best)

    raise RuntimeError(
        "Could not extract a links table from InteractionSpace. "
        "Tried .interaction_elements, then scanning attributes for long links or LR×cellpair matrices."
    )

def plot_circos_safe(
    interaction_space,
    label,
    group_meta,
    colors,
    sender_cells=None,
    receiver_cells=None,
    ligands=None,
    receptors=None,
    score_floor=0.0,
    max_senders=6,
    max_receivers=6,
    max_lr=12,
    complex_sep="&",
    debug=False,
):
    """
    Robust circos plot wrapper:
    - extracts available ligands/receptors (including complexes) from InteractionSpace
    - expands single genes to complexes when needed
    - if the requested subset is empty, auto-falls back to top interactions
    """
    import cell2cell as c2c  # relies on your env
    import matplotlib.pyplot as plt

    links = _find_links_df(interaction_space)

    # drop missing scores if any
    if "score" in links.columns:
        links = links.copy()
        links["score"] = pd.to_numeric(links["score"], errors="coerce").fillna(0.0)

    # apply score floor early for ranking
    links_f = links[links["score"] > float(score_floor)].copy()

    # expand ligands/receptors to complexes actually present
    avail_lig = links_f["ligand"].unique()
    avail_rec = links_f["receptor"].unique()
    ligands_exp = _expand_to_complexes(ligands, avail_lig, sep=complex_sep)
    receptors_exp = _expand_to_complexes(receptors, avail_rec, sep=complex_sep)

    # filter by user constraints
    if sender_cells is not None:
        links_f = links_f[links_f["sender"].isin(list(sender_cells))]
    if receiver_cells is not None:
        links_f = links_f[links_f["receiver"].isin(list(receiver_cells))]
    if ligands_exp is not None and len(ligands_exp) > 0:
        links_f = links_f[links_f["ligand"].isin(ligands_exp)]
    if receptors_exp is not None and len(receptors_exp) > 0:
        links_f = links_f[links_f["receptor"].isin(receptors_exp)]

    # If empty: automatically pick top LR and top senders/receivers under the (sender/receiver) constraints
    if links_f.empty:
        base = links[links["score"] > float(score_floor)].copy()
        if sender_cells is not None:
            base = base[base["sender"].isin(list(sender_cells))]
        if receiver_cells is not None:
            base = base[base["receiver"].isin(list(receiver_cells))]

        if base.empty:
            raise RuntimeError(
                "No interactions found above score_floor after applying sender/receiver constraints. "
                "Try lowering score_floor, or verify cell-type names match metadata."
            )

        # pick top LR pairs
        lr_rank = (base.groupby(["ligand", "receptor"])["score"]
                        .sum()
                        .sort_values(ascending=False)
                        .head(int(max_lr)))
        top_lr = set(lr_rank.index)
        base = base.set_index(["ligand", "receptor"])
        base = base.loc[list(top_lr)].reset_index()

        # pick top senders/receivers
        top_s = (base.groupby("sender")["score"].sum().sort_values(ascending=False).head(int(max_senders)).index.tolist())
        top_r = (base.groupby("receiver")["score"].sum().sort_values(ascending=False).head(int(max_receivers)).index.tolist())

        ligands_exp = base["ligand"].unique().tolist()
        receptors_exp = base["receptor"].unique().tolist()
        sender_cells = top_s
        receiver_cells = top_r

        if debug:
            print("[plot_circos_safe] Requested subset was empty -> auto-selected:")
            print("  senders:", sender_cells)
            print("  receivers:", receiver_cells)
            print("  #ligands:", len(ligands_exp), " #receptors:", len(receptors_exp))

    # Final plotting call uses cell2cell's own function (so legend/layout matches the library)
    ax = c2c.plotting.circos_plot(
        interaction_space=interaction_space,
        sender_cells=sender_cells,
        receiver_cells=receiver_cells,
        ligands=ligands_exp,
        receptors=receptors_exp,
        excluded_score=float(score_floor),
        metadata=group_meta,
        sample_col="Celltype",
        group_col="Group",
        colors=colors,
    )

# ---------- usage (replace your failing part) ----------

# keep your existing group_meta construction (must have columns: 'Celltype', 'Group')
# group_meta = ...

# If you still want to "try" your original manual subset first, pass it here:
sender_cells_f = ['fibroblast', 'endothelial cell of high endothelial venule', 'stromal cell']
receiver_cells_f = ['endothelial cell of high endothelial venule', 'classical monocyte', 'macrophage', 'stromal cell', 'fibroblast']
ligands_f = ['Fn1', 'Col4a1', 'Col4a2']
receptors_f = ['Cd74', 'Itgb1', 'Ncl']   # will be auto-expanded to complexes if needed

ax = plot_circos_safe(
    interaction_space=interactions_f,
    label="female",
    group_meta=group_meta,
    colors=colors,
    sender_cells=sender_cells_f,
    receiver_cells=receiver_cells_f,
    ligands=ligands_f,
    receptors=receptors_f,
    score_floor=0.0,
    max_senders=6,
    max_receivers=6,
    max_lr=12
)
plt.savefig(RESULTS_DIR / "cell2cell_circos_female.svg", bbox_inches="tight")
plt.show()

ax = plot_circos_safe(
    interaction_space=interactions_m,
    label="male",
    group_meta=group_meta,
    colors=colors,
    score_floor=0.0,
    top_edges=300,
    max_senders=6,
    max_receivers=6,
    max_lr=12
)
plt.savefig(RESULTS_DIR / "cell2cell_circos_male.svg", bbox_inches="tight")
plt.show()

# -------------------------
# 9) Plot: dot_plot for significance (communication + interactions)
# -------------------------
def plot_dotplots(interactions, rnaseq_data, meta, label):
    sender_cells, receiver_cells, _, _ = pick_sender_receiver_and_lr(rnaseq_data, meta)

    fig = c2c.plotting.dot_plot(
        interactions,
        evaluation="communication",
        significance=SIGNIF_FOR_DOTPLOT,
        figsize=(9, 14),
        cmap="PuOr",
        senders=sender_cells,
        receivers=receiver_cells,
        tick_size=8
    )
    plt.title(f"Dot plot (communication) — {label}", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f"cell2cell_dotplot_communication_{label}.png", dpi=300)
    plt.savefig(RESULTS_DIR / f"cell2cell_dotplot_communication_{label}.svg")
    plt.show()

    fig = c2c.plotting.dot_plot(
        interactions,
        evaluation="interactions",
        significance=SIGNIF_FOR_DOTPLOT,
        figsize=(12, 7),
        cmap="Blues",
    )
    plt.title(f"Dot plot (CCI/interaction potential) — {label}", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f"cell2cell_dotplot_interactions_{label}.png", dpi=300)
    plt.savefig(RESULTS_DIR / f"cell2cell_dotplot_interactions_{label}.svg")
    plt.show()

plot_dotplots(interactions_f, rnaseq_f, meta_f, "female")
plot_dotplots(interactions_m, rnaseq_m, meta_m, "male")

# -------------------------
# 10) Plot: clustermap_cci (undirected triangular) + legend
# -------------------------
def plot_clustermap_cci(interactions, label):
    fn = str(RESULTS_DIR / f"cell2cell_clustermap_cci_{label}.svg")
    cm = c2c.plotting.clustermap_cci(
        interactions,
        method="complete",
        metadata=group_meta,
        sample_col="Celltype",
        group_col="Group",
        colors=colors,
        title=f"CCI scores for cell-types ({label})",
        cmap="Blues",
        filename=fn
    )
    c2c.plotting.generate_legend(
        color_dict=colors,
        loc="center left",
        bbox_to_anchor=(1.15, 0.5),
        ncol=1,
        fancybox=True,
        shadow=True,
        title="Groups",
        fontsize=11,
    )
    plt.show()

plot_clustermap_cci(interactions_f, "female")
plot_clustermap_cci(interactions_m, "male")

print("\n[OK] Saved Cell2Cell-style figures in:", RESULTS_DIR.resolve())


In [ ]:
import re
import difflib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cell2cell as c2c

# -------------------------
# Basic helpers
# -------------------------
def _unwrap_space(obj):
    if hasattr(obj, "interaction_space") and getattr(obj, "interaction_space") is not None:
        return getattr(obj, "interaction_space")
    return obj

def _norm(s: str) -> str:
    return re.sub(r"\s+", " ", str(s).strip().lower())

def _fuzzy_map_list(requested, available, cutoff=0.6):
    if requested is None:
        return None, {"mapped": [], "unmatched": [], "suggestions": {}}
    requested = [str(x) for x in requested if x is not None]
    if len(requested) == 0:
        return [], {"mapped": [], "unmatched": [], "suggestions": {}}

    available = [str(x) for x in available]
    avail_norm = {_norm(x): x for x in available}
    keys = list(avail_norm.keys())

    mapped, unmatched, suggestions = [], [], {}
    for r in requested:
        rn = _norm(r)
        if rn in avail_norm:
            mapped.append(avail_norm[rn])
            continue
        close = difflib.get_close_matches(rn, keys, n=3, cutoff=cutoff)
        if close:
            mapped.append(avail_norm[close[0]])
            suggestions[r] = [avail_norm[c] for c in close]
        else:
            unmatched.append(r)
            suggestions[r] = []

    out, seen = [], set()
    for x in mapped:
        if x not in seen:
            out.append(x); seen.add(x)
    return out, {"mapped": out, "unmatched": unmatched, "suggestions": suggestions}

def _split_two_key(k):
    """Return (a,b) if k encodes a pair; else None."""
    if isinstance(k, (tuple, list)) and len(k) >= 2:
        return str(k[0]), str(k[1])
    if isinstance(k, str):
        seps = ["|", "→", "->", "__", ";", "::", "/"]
        for sep in seps:
            if sep in k:
                a, b = k.split(sep, 1)
                a, b = a.strip(), b.strip()
                if a and b:
                    return a, b
    return None

def _expand_to_complexes(requested_genes, available_symbols, sep="&"):
    """Expand requested gene to any complexes in available_symbols that contain it."""
    if requested_genes is None:
        return None
    requested_genes = [str(x) for x in requested_genes if x is not None]
    if len(requested_genes) == 0:
        return []

    avail = [str(a) for a in pd.unique(pd.Series(list(available_symbols)).dropna())]
    avail_lower = {a.lower(): a for a in avail}

    expanded = []
    for g in requested_genes:
        gl = g.lower()
        if gl in avail_lower:
            expanded.append(avail_lower[gl])
            continue
        hits = []
        for a in avail:
            parts = [p.strip().lower() for p in a.split(sep)]
            if gl in parts:
                hits.append(a)
        expanded.extend(hits)

    out, seen = [], set()
    for x in expanded:
        if x not in seen:
            out.append(x); seen.add(x)
    return out

# -------------------------
# Core: interpret interaction_elements["communication_matrix"]
# -------------------------
def _pick_comm_key(ie: dict):
    if "communication_matrix" in ie and isinstance(ie["communication_matrix"], pd.DataFrame):
        return "communication_matrix"
    # fallback: any key containing both words
    for k in ie.keys():
        if isinstance(ie[k], pd.DataFrame) and ("communication" in str(k).lower()) and ("matrix" in str(k).lower()):
            return k
    raise RuntimeError(f"No communication matrix DataFrame found in interaction_elements keys={list(ie.keys())}")

def _ensure_lr_index_from_pairs(df: pd.DataFrame, pairs):
    """
    Force df.index to MultiIndex(ligand,receptor).
    Priority:
      1) already MultiIndex
      2) df.index entries pair-like
      3) pairs list (same length as rows)
    """
    if isinstance(df.index, pd.MultiIndex) and df.index.nlevels >= 2:
        out = df.copy()
        out.index = pd.MultiIndex.from_tuples(
            [(str(a), str(b)) for a, b, *_ in out.index],
            names=["ligand", "receptor"]
        ) if out.index.nlevels > 2 else pd.MultiIndex.from_tuples(
            [(str(a), str(b)) for a, b in out.index],
            names=["ligand", "receptor"]
        )
        return out

    if len(df.index) > 0 and _split_two_key(df.index[0]) is not None:
        tups = []
        for x in df.index:
            ab = _split_two_key(x)
            if ab is None:
                raise RuntimeError(f"Index contains non-pair entry: {x!r}")
            tups.append((str(ab[0]), str(ab[1])))
        out = df.copy()
        out.index = pd.MultiIndex.from_tuples(tups, names=["ligand", "receptor"])
        return out

    if pairs is not None and isinstance(pairs, list) and len(pairs) == df.shape[0]:
        tups = []
        for p in pairs:
            ab = _split_two_key(p)
            if ab is None:
                # if p is tuple/list with >=2, split_two_key handles it; otherwise this is truly unsupported
                raise RuntimeError(f"pairs contains non-pair entry: {p!r}")
            tups.append((str(ab[0]), str(ab[1])))
        out = df.copy()
        out.index = pd.MultiIndex.from_tuples(tups, names=["ligand", "receptor"])
        return out

    # last resort: try to split string index with common separators
    seps = ["|", "→", "->", "__", ";", "::", "/"]
    idx = pd.Index(df.index.astype(str))
    for sep in seps:
        parts = idx.to_series().str.split(sep, n=1, expand=True)
        if isinstance(parts, pd.DataFrame) and parts.shape[1] >= 2:
            a = parts.iloc[:, 0].astype(str)
            b = parts.iloc[:, 1].astype(str)
            ok = (b.notna() & (b.str.len() > 0)).mean()
            if ok > 0.98:
                out = df.copy()
                out.index = pd.MultiIndex.from_arrays([a.values, b.values], names=["ligand", "receptor"])
                return out

    raise RuntimeError(
        "Could not build (ligand,receptor) MultiIndex for communication_matrix.\n"
        f"df.index example={list(map(str, df.index[:8]))}"
    )

def _ensure_cellpair_columns(df: pd.DataFrame, cell_names=None, cells=None):
    """
    Force df.columns to MultiIndex(sender,receiver).
    Handles:
      - already MultiIndex
      - tuple/list columns
      - columns are numeric 0..N-1 with a 'cells' dict mapping index->pair or pair->index
      - columns length == n_cells*n_cells using cell_names ordering
    """
    if isinstance(df.columns, pd.MultiIndex) and df.columns.nlevels >= 2:
        out = df.copy()
        out.columns = pd.MultiIndex.from_tuples(
            [(str(a), str(b)) for a, b, *_ in out.columns],
            names=["sender", "receiver"]
        ) if out.columns.nlevels > 2 else pd.MultiIndex.from_tuples(
            [(str(a), str(b)) for a, b in out.columns],
            names=["sender", "receiver"]
        )
        return out

    if len(df.columns) > 0 and _split_two_key(df.columns[0]) is not None:
        tups = []
        for x in df.columns:
            ab = _split_two_key(x)
            if ab is None:
                raise RuntimeError(f"Columns contain non-pair entry: {x!r}")
            tups.append((str(ab[0]), str(ab[1])))
        out = df.copy()
        out.columns = pd.MultiIndex.from_tuples(tups, names=["sender", "receiver"])
        return out

    # Try mapping via cells dict if it looks like an index map
    if isinstance(cells, dict) and len(cells) > 0:
        # case A: pair->int
        k0 = next(iter(cells.keys()))
        v0 = cells[k0]
        if _split_two_key(k0) is not None and isinstance(v0, (int, np.integer)):
            # build ordered list of length max+1
            n = int(max(cells.values())) + 1
            ordered = [None] * n
            for k, v in cells.items():
                ab = _split_two_key(k)
                if ab is None:
                    continue
                ordered[int(v)] = (str(ab[0]), str(ab[1]))
            if all(x is not None for x in ordered) and df.shape[1] == len(ordered):
                out = df.copy()
                out.columns = pd.MultiIndex.from_tuples(ordered, names=["sender", "receiver"])
                return out

        # case B: int->pair
        if isinstance(k0, (int, np.integer)) and _split_two_key(v0) is not None:
            n = int(max(cells.keys())) + 1
            ordered = [None] * n
            for k, v in cells.items():
                ab = _split_two_key(v)
                if ab is None:
                    continue
                ordered[int(k)] = (str(ab[0]), str(ab[1]))
            if all(x is not None for x in ordered) and df.shape[1] == len(ordered):
                out = df.copy()
                out.columns = pd.MultiIndex.from_tuples(ordered, names=["sender", "receiver"])
                return out

    # Try reconstructing from cell_names if columns is n_cells*n_cells
    if isinstance(cell_names, list) and len(cell_names) > 0:
        n = len(cell_names)
        if df.shape[1] == n * n:
            ordered = [(str(s), str(r)) for s in cell_names for r in cell_names]
            out = df.copy()
            out.columns = pd.MultiIndex.from_tuples(ordered, names=["sender", "receiver"])
            return out

    # last resort: split string columns
    seps = ["|", "→", "->", "__", ";", "::", "/"]
    cols = pd.Index(df.columns.astype(str))
    for sep in seps:
        parts = cols.to_series().str.split(sep, n=1, expand=True)
        if isinstance(parts, pd.DataFrame) and parts.shape[1] >= 2:
            a = parts.iloc[:, 0].astype(str)
            b = parts.iloc[:, 1].astype(str)
            ok = (b.notna() & (b.str.len() > 0)).mean()
            if ok > 0.98:
                out = df.copy()
                out.columns = pd.MultiIndex.from_arrays([a.values, b.values], names=["sender", "receiver"])
                return out

    raise RuntimeError(
        "Could not build (sender,receiver) MultiIndex for communication_matrix.\n"
        f"df.columns example={list(map(str, df.columns[:12]))}"
    )

def get_standardized_comm_matrix(interactions, verbose=True):
    """
    Returns a numeric DataFrame M with:
      - M.index = MultiIndex(ligand,receptor)
      - M.columns = MultiIndex(sender,receiver)
    built from interaction_elements['communication_matrix'] + siblings (pairs, cell_names, cells).
    """
    space = _unwrap_space(interactions)
    ie = getattr(space, "interaction_elements", None)
    if not isinstance(ie, dict):
        raise RuntimeError(f"interaction_elements is not a dict. type={type(ie)}")

    key = _pick_comm_key(ie)
    df = ie[key]
    if not isinstance(df, pd.DataFrame):
        raise RuntimeError(f"{key} is not a DataFrame. type={type(df)}")

    pairs = ie.get("pairs", None)
    cell_names = ie.get("cell_names", None)
    cells = ie.get("cells", None)

    if verbose:
        print(f"[get_standardized_comm_matrix] using key='{key}' shape={df.shape} "
              f"index={type(df.index).__name__} cols={type(df.columns).__name__}")
        if isinstance(pairs, list):
            print(f"  pairs: list len={len(pairs)}")
        if isinstance(cell_names, list):
            print(f"  cell_names: list len={len(cell_names)}")
        if isinstance(cells, dict):
            print(f"  cells: dict len={len(cells)}")

    M = df.copy()
    # numeric clean
    M = M.apply(pd.to_numeric, errors="coerce").fillna(0.0)

    # standardize axes
    M = _ensure_lr_index_from_pairs(M, pairs)
    M = _ensure_cellpair_columns(M, cell_names=cell_names, cells=cells)
    return M

# -------------------------
# Final circos wrapper using standardized comm matrix
# -------------------------
def plot_circos_cell2cell_robust_v81(
    interactions,
    label: str,
    group_meta: pd.DataFrame,
    colors: dict,
    sender_cells=None,
    receiver_cells=None,
    ligands=None,
    receptors=None,
    score_floor: float = 0.0,
    max_senders: int = 6,
    max_receivers: int = 6,
    max_lr: int = 12,
    complex_sep: str = "&",
    verbose: bool = True,
):
    if verbose:
        print(f"[plot_circos_cell2cell_robust_v81] {label}")

    space = _unwrap_space(interactions)
    M = get_standardized_comm_matrix(interactions, verbose=verbose)

    # --- HARD SAFETY: ensure standardized axes really are MultiIndex ---
    # (your debug shows comm_matrix starts as Index/Index; this forces MultiIndex again if needed)
    ie = getattr(space, "interaction_elements", None)
    if isinstance(ie, dict):
        if not (isinstance(M.index, pd.MultiIndex) and M.index.nlevels >= 2):
            M = _ensure_lr_index_from_pairs(M, ie.get("pairs", None))
        if not (isinstance(M.columns, pd.MultiIndex) and M.columns.nlevels >= 2):
            M = _ensure_cellpair_columns(M, cell_names=ie.get("cell_names", None), cells=ie.get("cells", None))

    if not (isinstance(M.columns, pd.MultiIndex) and M.columns.nlevels >= 2):
        raise RuntimeError(f"[{label}] Expected MultiIndex columns (sender,receiver) after standardization, got {type(M.columns)}")
    if not (isinstance(M.index, pd.MultiIndex) and M.index.nlevels >= 2):
        raise RuntimeError(f"[{label}] Expected MultiIndex index (ligand,receptor) after standardization, got {type(M.index)}")

    # apply score floor; if kills all edges and floor>0, relax to >0
    Mf = M.where(M > float(score_floor), 0.0)
    if Mf.values.max() <= 0 and float(score_floor) > 0:
        if verbose:
            print(f"[{label}] score_floor removed all edges; falling back to score_floor=0.")
        Mf = M.where(M > 0.0, 0.0)

    if Mf.values.max() <= 0:
        raise RuntimeError(f"[{label}] All communication scores are 0 (nothing to plot).")

    # available symbols
    all_senders = pd.Index(Mf.columns.get_level_values(0).astype(str)).unique().tolist()
    all_receivers = pd.Index(Mf.columns.get_level_values(1).astype(str)).unique().tolist()
    all_ligands = pd.Index(Mf.index.get_level_values(0).astype(str)).unique().tolist()
    all_receptors = pd.Index(Mf.index.get_level_values(1).astype(str)).unique().tolist()

    # map requested sender/receiver
    s_map, rep_s = _fuzzy_map_list(sender_cells, all_senders)
    r_map, rep_r = _fuzzy_map_list(receiver_cells, all_receivers)

    if verbose and sender_cells is not None:
        print(f"[{label}] sender requested={sender_cells}")
        print(f"[{label}] sender mapped={s_map}")
        if rep_s["unmatched"]:
            print(f"[{label}] sender unmatched={rep_s['unmatched']} suggestions={rep_s['suggestions']}")
    if verbose and receiver_cells is not None:
        print(f"[{label}] receiver requested={receiver_cells}")
        print(f"[{label}] receiver mapped={r_map}")
        if rep_r["unmatched"]:
            print(f"[{label}] receiver unmatched={rep_r['unmatched']} suggestions={rep_r['suggestions']}")

    # --- FIX: Index.isin returns numpy array already (no .to_numpy()) ---
    col_mask = np.ones(Mf.shape[1], dtype=bool)
    if s_map:
        col_mask &= np.asarray(Mf.columns.get_level_values(0).isin(s_map), dtype=bool)
    if r_map:
        col_mask &= np.asarray(Mf.columns.get_level_values(1).isin(r_map), dtype=bool)

    Mf_sr = Mf.iloc[:, col_mask]
    if Mf_sr.values.max() <= 0:
        if verbose:
            print(f"[{label}] No edges under sender/receiver constraints -> relaxing SR constraints.")
        Mf_sr = Mf

    # expand ligand/receptor requests to complexes if needed
    lig_exp = _expand_to_complexes(ligands, all_ligands, sep=complex_sep)
    rec_exp = _expand_to_complexes(receptors, all_receptors, sep=complex_sep)

    row_mask = np.ones(Mf_sr.shape[0], dtype=bool)
    if lig_exp is not None and len(lig_exp) > 0:
        row_mask &= np.asarray(Mf_sr.index.get_level_values(0).isin(lig_exp), dtype=bool)
    if rec_exp is not None and len(rec_exp) > 0:
        row_mask &= np.asarray(Mf_sr.index.get_level_values(1).isin(rec_exp), dtype=bool)

    Mf_srlr = Mf_sr.iloc[row_mask, :]
    if Mf_srlr.values.max() <= 0:
        if verbose:
            print(f"[{label}] Requested ligands/receptors yield 0 edges -> relaxing LR constraints.")
        Mf_srlr = Mf_sr

    # pick top LR rows
    row_strength = Mf_srlr.sum(axis=1).sort_values(ascending=False)
    top_lr_idx = row_strength.head(int(max_lr)).index
    Mf_show = Mf_srlr.loc[top_lr_idx]

    # rank senders/receivers within selected LR
    col_strength = Mf_show.sum(axis=0)
    sender_strength = col_strength.groupby(level=0).sum().sort_values(ascending=False)
    receiver_strength = col_strength.groupby(level=1).sum().sort_values(ascending=False)

    sender_use = sender_strength.head(int(max_senders)).index.astype(str).tolist()
    receiver_use = receiver_strength.head(int(max_receivers)).index.astype(str).tolist()

    lig_use = pd.Index(top_lr_idx.get_level_values(0).astype(str)).unique().tolist()
    rec_use = pd.Index(top_lr_idx.get_level_values(1).astype(str)).unique().tolist()

    if verbose:
        print(f"[{label}] circos senders={sender_use}")
        print(f"[{label}] circos receivers={receiver_use}")
        print(f"[{label}] circos ligands={lig_use}")
        print(f"[{label}] circos receptors={rec_use}")

    ax = c2c.plotting.circos_plot(
        interaction_space=space,   # InteractionSpace
        sender_cells=sender_use,
        receiver_cells=receiver_use,
        ligands=lig_use,
        receptors=rec_use,
        excluded_score=float(score_floor),
        metadata=group_meta,
        sample_col="Celltype",
        group_col="Group",
        colors=colors,
        fontsize=14,
    )
    plt.title(f"Circos ({label})", fontsize=14)
    return ax


In [ ]:
ax = plot_circos_cell2cell_robust_v81(
    interactions=interactions_f_ct,
    label="female",
    group_meta=group_meta,
    colors=colors,
    sender_cells=sender_cells_f,
    receiver_cells=receiver_cells_f,
    ligands=ligands_f,
    receptors=receptors_f,
    score_floor=0.0,
    max_senders=6,
    max_receivers=6,
    max_lr=12,
    verbose=True,
)
plt.savefig(RESULTS_DIR / "cell2cell_circos_female.svg", bbox_inches="tight")
plt.show()


In [ ]:
# -------------------------
# 2) Load Mouse Jin LR pairs + ROBUST map to your var_names
# -------------------------
import numpy as np
import pandas as pd
from pathlib import Path

LR_CACHE = RESULTS_DIR / "Mouse-2020-Jin-LR-pairs.csv"
LR_URL   = "https://raw.githubusercontent.com/LewisLabUCSD/Ligand-Receptor-Pairs/master/Mouse/Mouse-2020-Jin-LR-pairs.csv"

def split_complex(s, sep="&"):
    return [x.strip() for x in str(s).split(sep) if x.strip()]

def load_mouse_lr_pairs(cache_path: Path, url: str) -> pd.DataFrame:
    if cache_path.exists():
        df = pd.read_csv(cache_path)
        print(f"[OK] Loaded LR pairs from cache -> {cache_path} | rows={len(df)}")
    else:
        df = pd.read_csv(url)
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(cache_path, index=False)
        print(f"[OK] Downloaded LR pairs -> {cache_path} | rows={len(df)}")

    if "ligand_symbol" not in df.columns or "receptor_symbol" not in df.columns:
        cand_l = [c for c in df.columns if "ligand" in c.lower()]
        cand_r = [c for c in df.columns if "receptor" in c.lower()]
        if not cand_l or not cand_r:
            raise KeyError(f"Unexpected LR columns: {df.columns.tolist()}")
        df = df.rename(columns={cand_l[0]: "ligand_symbol", cand_r[0]: "receptor_symbol"})

    return df[["ligand_symbol", "receptor_symbol"]].astype(str)

def _norm_gene(x: str) -> str:
    x = str(x).strip()
    if x == "" or x.lower() in {"nan", "none", "null"}:
        return ""
    return x.lower()

def build_varmap_first(varnames: pd.Index) -> dict:
    """
    Map normalized symbol -> a concrete var_name (first occurrence).
    This avoids subtle failures due to casing/whitespace and make_unique suffixes.
    """
    m = {}
    for g in varnames.astype(str).tolist():
        k = _norm_gene(g)
        if not k:
            continue
        if k not in m:
            m[k] = g  # keep first
    return m

lr_df = load_mouse_lr_pairs(LR_CACHE, LR_URL)

varmap = build_varmap_first(ad_sym.var_names)
print("[INFO] varmap size (non-empty symbols):", len(varmap))

def map_lr_pairs_to_varnames(lr_df: pd.DataFrame, varmap: dict, sep="&") -> pd.DataFrame:
    out_rows = []
    missing = 0
    for lig, rec in zip(lr_df["ligand_symbol"].values, lr_df["receptor_symbol"].values):
        lig_parts = split_complex(lig, sep=sep)
        rec_parts = split_complex(rec, sep=sep)

        lig_m = []
        ok = True
        for p in lig_parts:
            k = _norm_gene(p)
            if (not k) or (k not in varmap):
                ok = False; break
            lig_m.append(varmap[k])
        if not ok:
            missing += 1
            continue

        rec_m = []
        for p in rec_parts:
            k = _norm_gene(p)
            if (not k) or (k not in varmap):
                ok = False; break
            rec_m.append(varmap[k])
        if not ok:
            missing += 1
            continue

        out_rows.append(("&".join(lig_m), "&".join(rec_m)))

    out = pd.DataFrame(out_rows, columns=["ligand_symbol", "receptor_symbol"]).drop_duplicates().reset_index(drop=True)
    return out, missing

lr_pairs_mapped, n_missing_pairs = map_lr_pairs_to_varnames(lr_df, varmap, sep="&")
print("[INFO] LR pairs mapped to your genes:", len(lr_pairs_mapped), "| dropped (missing symbol):", n_missing_pairs)

# -------------------------
# 2b) Build keep_idx from USED LR GENES (robust) + sanity checks
# -------------------------
used_lr_genes = set()
for col in ["ligand_symbol", "receptor_symbol"]:
    for s in lr_pairs_mapped[col].astype(str).values:
        for g in split_complex(s, "&"):
            used_lr_genes.add(str(g).strip())

used_lr_genes = sorted(used_lr_genes)
idx = ad_sym.var_names.get_indexer(used_lr_genes)
missing_genes = [g for g, i in zip(used_lr_genes, idx) if i < 0]
keep_idx = idx[idx >= 0]
keep_idx = np.unique(keep_idx)

print("[INFO] Used LR genes (expanded):", len(used_lr_genes))
print("[INFO] LR genes FOUND in var_names:", len(keep_idx))
print("[INFO] Missing LR genes (should be ~0):", len(missing_genes))
if len(missing_genes) > 0:
    print("  examples missing:", missing_genes[:20])

# hard stop if something is still off
if len(keep_idx) < 200:
    raise RuntimeError(
        f"Too few LR genes found in var_names after robust mapping: {len(keep_idx)}.\n"
        "This indicates gene-symbol mismatch. Check ad_sym.var_names examples:\n"
        f"  var_names head: {ad_sym.var_names[:10].tolist()}\n"
        f"  example LR gene symbols: {lr_df['ligand_symbol'].head(10).tolist()}\n"
    )

# -------------------------
# 2c) Build Squidpy interactions from mapped LR pairs (expanded)
# -------------------------
def build_sq_interactions_df_from_mapped(lr_pairs_mapped: pd.DataFrame, sep="&") -> pd.DataFrame:
    rows = []
    for lig, rec in zip(lr_pairs_mapped["ligand_symbol"].values, lr_pairs_mapped["receptor_symbol"].values):
        ligs = split_complex(lig, sep=sep)
        recs = split_complex(rec, sep=sep)
        for a in ligs:
            for b in recs:
                rows.append((a, b))
    inter = pd.DataFrame(rows, columns=["source", "target"]).drop_duplicates().reset_index(drop=True)
    return inter

SQ_INTERACTIONS = build_sq_interactions_df_from_mapped(lr_pairs_mapped, sep="&")
# filter to genes present (should keep almost all)
SQ_INTERACTIONS = SQ_INTERACTIONS[
    SQ_INTERACTIONS["source"].isin(ad_sym.var_names) & SQ_INTERACTIONS["target"].isin(ad_sym.var_names)
].copy()

print("[INFO] Squidpy interactions (expanded, filtered):", SQ_INTERACTIONS.shape)
if SQ_INTERACTIONS.empty:
    raise RuntimeError("SQ_INTERACTIONS is empty even after robust mapping — something is wrong with gene symbols.")

import copy
import inspect
import pandas as pd

def _sanitize_df_for_reset_index(df: pd.DataFrame, tag: str) -> pd.DataFrame:
    df = df.copy()

    # Ensure index/columns have names (prevents pandas from inventing level_0/level_1)
    if df.index.name is None:
        df.index.name = f"{tag}_idx"
    if df.columns.name is None:
        df.columns.name = f"{tag}_col"

    # If MultiIndex, ensure every level has a non-empty, non-colliding name
    if isinstance(df.index, pd.MultiIndex):
        names = []
        used = set(map(str, df.columns))
        for i, n in enumerate(df.index.names):
            base = str(n).strip() if (n is not None and str(n).strip() != "") else f"{tag}_i{i}"
            if base in used:
                base = f"{base}_idx"
            names.append(base)
        df.index = df.index.set_names(names)

    if isinstance(df.columns, pd.MultiIndex):
        names = []
        used = set(map(str, df.columns.get_level_values(0)))  # just for collision avoidance
        for i, n in enumerate(df.columns.names):
            base = str(n).strip() if (n is not None and str(n).strip() != "") else f"{tag}_c{i}"
            if base in used:
                base = f"{base}_col"
            names.append(base)
        df.columns = df.columns.set_names(names)

    # If something already has level_0/level_1 as real columns, rename them away
    ren = {}
    if "level_0" in df.columns:
        ren["level_0"] = f"__level_0__{tag}"
    if "level_1" in df.columns:
        ren["level_1"] = f"__level_1__{tag}"
    if ren:
        df = df.rename(columns=ren)

    return df


def prepare_cell2cell_for_permutation(interactions, verbose=True):
    """
    Best-effort sanitization of interaction_elements matrices so pandas doesn't create level_0/level_1 collisions.
    """
    sp = getattr(interactions, "interaction_space", None)
    if sp is None:
        return

    ie = getattr(sp, "interaction_elements", None)
    if not isinstance(ie, dict):
        return

    for k, v in list(ie.items()):
        if isinstance(v, pd.DataFrame):
            ie[k] = _sanitize_df_for_reset_index(v, tag=str(k))
            if verbose:
                print(f"[sanitize] {k}: shape={ie[k].shape} index={type(ie[k].index).__name__} cols={type(ie[k].columns).__name__}")


def permute_cell_labels_safe(interactions, N=50, seed=0, label="", verbose=True):
    """
    Runs cell2cell permute_cell_labels on a deep-copied, sanitized interactions object.
    Returns: (interactions_perm, ok_bool)
    """
    if not hasattr(interactions, "permute_cell_labels"):
        if verbose:
            print(f"[WARN] {label}: interactions has no permute_cell_labels().")
        return interactions, False

    inter = copy.deepcopy(interactions)
    prepare_cell2cell_for_permutation(inter, verbose=verbose)

    fn = inter.permute_cell_labels
    sig = inspect.signature(fn)
    kwargs = {}
    if "N" in sig.parameters:
        kwargs["N"] = int(N)
    elif "n_perms" in sig.parameters:
        kwargs["n_perms"] = int(N)
    elif "n_permutations" in sig.parameters:
        kwargs["n_permutations"] = int(N)

    if "random_state" in sig.parameters:
        kwargs["random_state"] = int(seed)
    elif "seed" in sig.parameters:
        kwargs["seed"] = int(seed)

    try:
        if verbose:
            print(f"[{label}] permute_cell_labels kwargs={kwargs}")
        fn(**kwargs)
        return inter, True
    except Exception as e:
        if verbose:
            print(f"[WARN] {label}: permute_cell_labels failed -> {repr(e)}")
            print("       Continuing without significance filtering for plots.")
        return interactions, False



In [ ]:
sender_cells = sender_cells_f
receiver_cells = receiver_cells_f

In [ ]:
import copy
import inspect
import re
from contextlib import contextmanager
import pandas as pd

@contextmanager
def _patch_reset_index_level_collision(verbose=False):
    """
    Patch pandas.DataFrame.reset_index so that if it raises:
      ValueError('cannot insert level_0, already exists')
    we rename the existing 'level_0' (or whatever column) and retry.
    This is scoped to the context manager.
    """
    orig = pd.DataFrame.reset_index

    def patched(self, *args, **kwargs):
        try:
            return orig(self, *args, **kwargs)
        except ValueError as e:
            msg = str(e)
            m = re.search(r"cannot insert ([^,]+), already exists", msg)
            if m:
                col = m.group(1).strip()
                if col in self.columns:
                    # rename away and retry
                    newcol = f"__{col}__existing"
                    if verbose:
                        print(f"[patch_reset_index] renaming conflicting column {col!r} -> {newcol!r}")
                    tmp = self.rename(columns={col: newcol})
                    return orig(tmp, *args, **kwargs)
            raise

    pd.DataFrame.reset_index = patched
    try:
        yield
    finally:
        pd.DataFrame.reset_index = orig


def permute_cell_labels_safe_v2(interactions, N=50, seed=0, label="", verbose=True):
    """
    Runs cell2cell permute_cell_labels with a scoped pandas patch for the common
    'level_0 already exists' reset_index collision.
    Returns: (interactions_perm, ok_bool)
    """
    if not hasattr(interactions, "permute_cell_labels"):
        if verbose:
            print(f"[WARN] {label}: interactions has no permute_cell_labels().")
        return interactions, False

    # deep copy so we don't mutate your main object
    inter = copy.deepcopy(interactions)

    fn = inter.permute_cell_labels
    sig = inspect.signature(fn)

    # Build kwargs if possible
    kwargs = {}
    if "random_state" in sig.parameters:
        kwargs["random_state"] = int(seed)
    elif "seed" in sig.parameters:
        kwargs["seed"] = int(seed)

    # Prefer to pass N in kwargs if present; otherwise pass as first positional
    pass_N_as_kw = any(k in sig.parameters for k in ["N", "n_perms", "n_permutations", "n_permut"])
    if "N" in sig.parameters:
        kwargs["N"] = int(N)
    elif "n_perms" in sig.parameters:
        kwargs["n_perms"] = int(N)
    elif "n_permutations" in sig.parameters:
        kwargs["n_permutations"] = int(N)
    elif "n_permut" in sig.parameters:
        kwargs["n_permut"] = int(N)

    try:
        if verbose:
            print(f"[{label}] permute_cell_labels signature={sig}")
            print(f"[{label}] permute_cell_labels kwargs={kwargs}")

        with _patch_reset_index_level_collision(verbose=verbose):
            if pass_N_as_kw:
                fn(**kwargs)
            else:
                # N not discoverable as kw → try positional
                if verbose:
                    print(f"[{label}] passing N as positional={int(N)}")
                fn(int(N), **kwargs)

        return inter, True

    except Exception as e:
        if verbose:
            print(f"[WARN] {label}: permute_cell_labels failed -> {repr(e)}")
            print("       Continuing without significance filtering for plots.")
        return interactions, False


In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cell2cell as c2c

# -------------------------
# robust helpers
# -------------------------
def _as_array(x):
    if isinstance(x, pd.Index):
        return x.to_numpy()
    return np.asarray(x)

def _isin_mask(x, values):
    if values is None or len(values) == 0:
        return np.ones(len(x), dtype=bool)
    return np.isin(_as_array(x).astype(object), np.asarray(list(values), dtype=object))

def _parse_pair_label(lbl):
    s = str(lbl)
    seps = [" --> ", "-->", " -> ", "->", "→", ";", "|", "__", ","]
    for sep in seps:
        if sep in s:
            a, b = s.split(sep, 1)
            a, b = a.strip(), b.strip()
            if a and b:
                return a, b
    m = re.match(r"^\(?\s*([^,]+?)\s*,\s*([^)]+?)\s*\)?$", s)
    if m:
        return m.group(1).strip(), m.group(2).strip()
    return None

def get_standardized_comm_matrix(interactions, verbose=True):
    ie = interactions.interaction_space.interaction_elements
    if "communication_matrix" not in ie:
        raise RuntimeError("interaction_elements has no key 'communication_matrix'.")
    M = ie["communication_matrix"]
    if not isinstance(M, pd.DataFrame):
        raise RuntimeError(f"communication_matrix is not a DataFrame: {type(M)}")

    # Try to infer cell_names (helps decode int-coded columns)
    cell_names = None
    if isinstance(ie.get("cell_names", None), list) and len(ie["cell_names"]) > 0:
        cell_names = [str(x) for x in ie["cell_names"]]
    elif isinstance(ie.get("cci_matrix", None), pd.DataFrame) and ie["cci_matrix"].shape[0] > 0:
        cell_names = [str(x) for x in ie["cci_matrix"].index]
    elif isinstance(ie.get("cells", None), dict) and len(ie["cells"]) > 0:
        keys = list(ie["cells"].keys())
        vals = list(ie["cells"].values())
        if all(isinstance(k, str) for k in keys) and all(isinstance(v, (int, np.integer)) for v in vals):
            cell_names = [k for k, _ in sorted(ie["cells"].items(), key=lambda kv: kv[1])]
        elif all(isinstance(k, (int, np.integer)) for k in keys) and all(isinstance(v, str) for v in vals):
            cell_names = [v for _, v in sorted(ie["cells"].items(), key=lambda kv: kv[0])]

    # already MultiIndex
    if isinstance(M.columns, pd.MultiIndex) and M.columns.nlevels >= 2:
        if verbose:
            print(f"[get_standardized_comm_matrix] columns already MultiIndex: {M.shape}")
        return M

    cols = list(M.columns)

    # int-coded columns: 0..n*n-1 (common in cell2cell internals)
    if cell_names is not None and len(cols) == len(cell_names) * len(cell_names) \
       and all(isinstance(c, (int, np.integer)) for c in cols):
        n = len(cell_names)
        send = [cell_names[int(c) // n] for c in cols]
        recv = [cell_names[int(c) %  n] for c in cols]
        M2 = M.copy()
        M2.columns = pd.MultiIndex.from_arrays([send, recv], names=["sender", "receiver"])
        if verbose:
            print(f"[get_standardized_comm_matrix] decoded int columns -> MultiIndex: {M2.shape}")
        return M2

    # string-coded columns like "A->B"
    parsed = [_parse_pair_label(c) for c in cols]
    if all(p is not None for p in parsed):
        send = [p[0] for p in parsed]
        recv = [p[1] for p in parsed]
        M2 = M.copy()
        M2.columns = pd.MultiIndex.from_arrays([send, recv], names=["sender", "receiver"])
        if verbose:
            print(f"[get_standardized_comm_matrix] parsed string columns -> MultiIndex: {M2.shape}")
        return M2

    if verbose:
        print(f"[get_standardized_comm_matrix] WARNING: could not convert columns to MultiIndex. cols example={cols[:5]}")
    return M

def choose_axes_from_comm_matrix(M, senders=None, receivers=None, top_lr=60, max_cellpairs=60):
    if not (isinstance(M.columns, pd.MultiIndex) and M.columns.nlevels >= 2):
        col_strength = M.sum(axis=0).sort_values(ascending=False)
        cols_keep = col_strength.head(min(max_cellpairs, len(col_strength))).index
        row_strength = M.loc[:, cols_keep].sum(axis=1).sort_values(ascending=False)
        rows_keep = row_strength.head(min(top_lr, len(row_strength))).index
        return rows_keep, cols_keep

    col_mask = np.ones(M.shape[1], dtype=bool)
    if senders:
        col_mask &= _isin_mask(M.columns.get_level_values(0), senders)
    if receivers:
        col_mask &= _isin_mask(M.columns.get_level_values(1), receivers)

    Msr = M.loc[:, col_mask]
    if Msr.shape[1] == 0:
        Msr = M

    col_strength = Msr.sum(axis=0).sort_values(ascending=False)
    cols_keep = col_strength.head(min(int(max_cellpairs), len(col_strength))).index

    sub = Msr.loc[:, cols_keep]
    row_strength = sub.sum(axis=1)
    if float(row_strength.max()) <= 0:
        row_strength = sub.max(axis=1)
    row_strength = row_strength.sort_values(ascending=False)
    rows_keep = row_strength.head(min(int(top_lr), len(row_strength))).index
    return rows_keep, cols_keep
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import cell2cell as c2c

# ---------- colormap helpers ----------
def truncate_cmap(cmap, minval=0.0, maxval=1.0, n=256):
    """Return a truncated version of a colormap."""
    cmap_obj = mpl.cm.get_cmap(cmap) if isinstance(cmap, str) else cmap
    colors = cmap_obj(np.linspace(minval, maxval, n))
    return mpl.colors.LinearSegmentedColormap.from_list(
        f"{cmap_obj.name}_trunc_{minval:.2f}_{maxval:.2f}", colors
    )

def smart_cmap_and_norm(vals, cmap="PuOr", gamma=0.65, truncate_diverging=True):
    """
    For positive-only vals, avoid diverging white midpoints by truncating diverging cmaps.
    Also use PowerNorm to make small signals visible.
    """
    vals = np.asarray(vals, dtype=float)
    vals = vals[np.isfinite(vals)]
    vals_pos = vals[vals > 0]

    if vals_pos.size == 0:
        cm = mpl.cm.get_cmap("viridis")
        norm = mpl.colors.Normalize(vmin=0, vmax=1)
        return cm, norm

    # robust range for color scaling
    vmin = float(np.quantile(vals_pos, 0.05))
    vmax = float(np.quantile(vals_pos, 0.995))
    if not np.isfinite(vmin): vmin = float(vals_pos.min())
    if not np.isfinite(vmax): vmax = float(vals_pos.max())
    if vmax <= vmin:
        vmax = float(vals_pos.max())
        vmin = float(vals_pos.min())

    diverging = {
        "PuOr","PRGn","BrBG","PiYG","RdBu","RdGy","RdYlBu","RdYlGn","Spectral","coolwarm","bwr","seismic"
    }

    if truncate_diverging and (vals.min() >= 0) and isinstance(cmap, str) and (cmap in diverging):
        # Remove the white midpoint:
        # PuOr is purple<->white<->orange, for positive-only data we keep the orange half.
        cm = truncate_cmap(cmap, 0.55, 1.0)
    else:
        cm = mpl.cm.get_cmap(cmap) if isinstance(cmap, str) else cmap

    norm = mpl.colors.PowerNorm(gamma=gamma, vmin=vmin, vmax=vmax, clip=True)
    return cm, norm

def compute_global_vmax_on_axes(
    interactions_list,    # list like [("all", inter_all), ("female", inter_f), ("male", inter_m)]
    rows_keep,
    cols_keep,
    log1p_scale=True,
):
    vmax = 0.0
    for lab, inter in interactions_list:
        if inter is None:
            continue
        M = get_standardized_comm_matrix(inter, verbose=False)
        M = M.apply(pd.to_numeric, errors="coerce").fillna(0.0)
        Msub = M.reindex(index=rows_keep, columns=cols_keep).fillna(0.0)
        V = Msub.to_numpy(dtype=float)
        if log1p_scale:
            V = np.log1p(V)
        vmax = max(vmax, float(np.nanmax(V)))
    return vmax

# ---------- UPDATED fallback dotplot ----------
def dot_plot_safe_v6(
    interactions,
    label: str,
    senders=None,
    receivers=None,
    significance=None,
    figsize=(9, 16),
    cmap="PuOr",
    tick_size=8,
    top_lr=60,
    max_cellpairs=60,
    fixed_rows=None,
    fixed_cols=None,
    log1p_scale=True,
    verbose=True,
    gamma=0.65,
    truncate_diverging=True,

    # NEW:
    force_color_0_1=True,     # <- force the colorbar to be 0..1
    global_vmax=None,         # <- provide shared vmax (after log1p if log1p_scale=True)
    size_by_scaled=True,      # <- size uses same scaled values so size contrast matches color
):
    # 1) try official cell2cell dot plot
    try:
        fig = c2c.plotting.dot_plot(
            interactions,
            evaluation="communication",
            significance=significance,
            figsize=figsize,
            cmap=cmap,
            senders=senders,
            receivers=receivers,
            tick_size=tick_size
        )
        return fig, None, None
    except Exception as e:
        if verbose:
            print(f"[WARN] cell2cell.dot_plot failed for {label}: {repr(e)}")
            print("       Falling back to custom dot plot from communication_matrix.")

    # 2) fallback matrix
    M = get_standardized_comm_matrix(interactions, verbose=verbose)
    M = M.apply(pd.to_numeric, errors="coerce").fillna(0.0)

    # choose axes
    if fixed_rows is None or fixed_cols is None:
        rows_keep, cols_keep = choose_axes_from_comm_matrix(
            M, senders=senders, receivers=receivers,
            top_lr=top_lr, max_cellpairs=max_cellpairs
        )
    else:
        rows_keep, cols_keep = fixed_rows, fixed_cols

    # reindex -> missing rows/cols become 0 (avoids KeyError across sex strata)
    Msub = M.reindex(index=rows_keep, columns=cols_keep).fillna(0.0)
    V = Msub.to_numpy(dtype=float)
    if log1p_scale:
        Vplot = np.log1p(V)
        val_label = "log1p(score)"
    else:
        Vplot = V
        val_label = "score"

    if not np.isfinite(Vplot).any() or float(np.nanmax(Vplot)) <= 0:
        raise RuntimeError(
            f"[{label}] fallback dot plot: selected submatrix is empty/all-zero. "
            f"Increase top_lr/max_cellpairs or relax sender/receiver filters."
        )

    # labels
    if isinstance(Msub.columns, pd.MultiIndex) and Msub.columns.nlevels >= 2:
        x_labels = [f"{s}→{r}" for s, r in Msub.columns.to_list()]
    else:
        x_labels = [str(c) for c in Msub.columns]
    y_labels = [str(i) for i in Msub.index]

    xs, ys = np.meshgrid(np.arange(Vplot.shape[1]), np.arange(Vplot.shape[0]))
    xs, ys = xs.ravel(), ys.ravel()
    vals = Vplot.ravel()

    nz = vals > 0
    if nz.sum() == 0:
        raise RuntimeError(f"[{label}] fallback dot plot: no nonzero values after scaling/subsetting.")

    # keep more points (so it doesn't look empty)
    thr = np.quantile(vals[nz], 0.40)
    keep = vals >= thr
    xs, ys, vals = xs[keep], ys[keep], vals[keep]

    # -------- NEW: force 0..1 color scaling (shared across plots) --------
    if force_color_0_1:
        if global_vmax is None:
            global_vmax = float(np.nanmax(Vplot))
        global_vmax = max(global_vmax, 1e-12)
        cvals = np.clip(vals / global_vmax, 0.0, 1.0)

        # fixed norm 0..1, plus gamma for better contrast
        norm = mpl.colors.PowerNorm(gamma=gamma, vmin=0.0, vmax=1.0, clip=True)

        # keep your "PuOr" but avoid the white midpoint for positive-only
        cm, _ = smart_cmap_and_norm(
            np.array([0.0, 1.0]), cmap=cmap, gamma=gamma, truncate_diverging=truncate_diverging
        )

        colorbar_label = f"scaled {val_label} (0–1)"
    else:
        # old behavior: data-driven norm
        cm, norm = smart_cmap_and_norm(vals, cmap=cmap, gamma=gamma, truncate_diverging=truncate_diverging)
        cvals = vals
        colorbar_label = val_label

    # sizes: use scaled values for better visual separation (optional)
    if size_by_scaled and force_color_0_1:
        vmaxv = float(cvals.max())
        sizes = 35 + 260 * (cvals / max(1e-12, vmaxv))
    else:
        vmaxv = float(vals.max())
        sizes = 35 + 260 * (vals / max(1e-12, vmaxv))

    fig, ax = plt.subplots(figsize=figsize)
    sc = ax.scatter(xs, ys, c=cvals, s=sizes, cmap=cm, norm=norm, alpha=0.95, linewidths=0)

    ax.set_xticks(np.arange(len(x_labels)))
    ax.set_xticklabels(x_labels, rotation=90, fontsize=tick_size)
    ax.set_yticks(np.arange(len(y_labels)))
    ax.set_yticklabels(y_labels, fontsize=tick_size)

    ax.set_title(f"DotPlot fallback — {label} (cellpairs={len(x_labels)}, LR={len(y_labels)})", fontsize=12)
    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label(colorbar_label, rotation=90)

    plt.tight_layout()
    return fig, rows_keep, cols_keep

# -------------------------
# FIX: resolve your interaction objects (solves NameError)
# -------------------------
def resolve_interactions(key):
    """
    Tries to find an InteractionSpace object for 'all'/'female'/'male'
    from common variable names or dict containers.
    """
    candidates = {
        "all":    ["interactions_all_ct", "interactions_all", "interactions"],
        "female": ["interactions_f_ct", "interactions_female_ct", "interactions_female", "interactions_f"],
        "male":   ["interactions_m_ct", "interactions_male_ct", "interactions_male", "interactions_m"],
    }[key]

    for name in candidates:
        obj = globals().get(name, None)
        if obj is not None:
            return obj, name

    for dname in ["interactions_by", "interactions_map"]:
        d = globals().get(dname, None)
        if isinstance(d, dict):
            for k in [key, key.upper(), key.capitalize()]:
                if k in d and d[k] is not None:
                    return d[k], f"{dname}['{k}']"

    return None, None

interactions_all_ct, src_all = resolve_interactions("all")
interactions_f_ct,   src_f   = resolve_interactions("female")
interactions_m_ct,   src_m   = resolve_interactions("male")

print("[resolve] all   :", src_all)
print("[resolve] female:", src_f)
print("[resolve] male  :", src_m)

if interactions_all_ct is None:
    raise RuntimeError(
        "Could not find ALL interactions object. "
        "Define one of: interactions_all_ct = <your_all_object>  OR  interactions = <your_all_object>."
    )

# also make sender_cells/receiver_cells optional if not defined
sender_cells  = globals().get("sender_cells", None)
receiver_cells = globals().get("receiver_cells", None)

# -------------------------
# Run dotplots with shared axes
# -------------------------
# ALL defines the axes
# 1) ALL defines axes
fig_all, rows_keep, cols_keep = dot_plot_safe_v6(
    interactions_all_ct, "all",
    senders=sender_cells, receivers=receiver_cells,
    top_lr=60, max_cellpairs=60,
    force_color_0_1=True,   # will use global_vmax later if provided
    global_vmax=None,       # ok for the first call
    verbose=True
)

# 2) compute shared vmax on those SAME axes (after log1p if enabled)
inter_list = [("all", interactions_all_ct), ("female", interactions_f_ct), ("male", interactions_m_ct)]
vmax_shared = compute_global_vmax_on_axes(inter_list, rows_keep, cols_keep, log1p_scale=True)

# 3) replot ALL + plot female/male with identical 0..1 scaling
fig_all, _, _ = dot_plot_safe_v6(
    interactions_all_ct, "all",
    senders=sender_cells, receivers=receiver_cells,
    fixed_rows=rows_keep, fixed_cols=cols_keep,
    force_color_0_1=True,
    global_vmax=vmax_shared,
    verbose=True
)

fig_f, _, _ = dot_plot_safe_v6(
    interactions_f_ct, "female",
    senders=sender_cells, receivers=receiver_cells,
    fixed_rows=rows_keep, fixed_cols=cols_keep,
    force_color_0_1=True,
    global_vmax=vmax_shared,
    verbose=True
)

fig_m, _, _ = dot_plot_safe_v6(
    interactions_m_ct, "male",
    senders=sender_cells, receivers=receiver_cells,
    fixed_rows=rows_keep, fixed_cols=cols_keep,
    force_color_0_1=True,
    global_vmax=vmax_shared,
    verbose=True
)


In [ ]:
# ============================================================
# PATCH: fix mouse protein resource mismatch + backed mode copy
# ============================================================

import re
import numpy as np
import pandas as pd
import scanpy as sc
import liana as li
import cell2cell as c2c

import decoupler as dc
from mudata import MuData
from scipy import sparse
from pathlib import Path
import matplotlib.pyplot as plt


# -------------------------
# Helpers
# -------------------------
def looks_like_mouse_symbols(gene_names):
    g = pd.Series(gene_names).astype(str)
    if len(g) == 0:
        return False
    mouse_like = g.str.match(r"^[A-Z][a-z0-9\-\.]+$").mean()
    human_like = g.str.match(r"^[A-Z0-9\-\.]+$").mean()
    return (mouse_like > 0.55) and (human_like < 0.55)

def human_to_mouse_symbol(s):
    s = str(s)
    if s == "" or s.lower() == "nan":
        return s
    return s[0] + s[1:].lower()

def detect_obs_key(adata, candidates, required=True, label="key"):
    for k in candidates:
        if k in adata.obs.columns:
            return k
    if required:
        raise KeyError(f"Could not detect {label}. Tried: {candidates}. obs has: {list(adata.obs.columns)[:80]}")
    return None

def detect_var_symbol_key(adata):
    for k in ["gene_symbols", "gene_symbol", "symbol", "gene", "gene_name", "gene_names"]:
        if k in adata.var.columns:
            return k
    return None

def ensure_sparse_float32(adata):
    X = adata.X
    if sparse.issparse(X):
        if not sparse.isspmatrix_csr(X):
            X = X.tocsr()
        if X.dtype != np.float32:
            X = X.astype(np.float32)
        adata.X = X
    else:
        # LIANA can work with dense, but for memory safety cast and sparsify if huge
        adata.X = np.asarray(X, dtype=np.float32)
    return adata

def make_gene_symbol_view_inplace(adata):
    """
    Prefer gene symbols in var_names for LIANA matching.
    Edits adata in place (no extra copy).
    """
    sym_key = detect_var_symbol_key(adata)
    if sym_key is None:
        return None  # already fine

    syms = adata.var[sym_key].astype(str).replace("nan", "").replace("None", "")
    syms = syms.where(syms.str.len() > 0, other=pd.Index(adata.var_names).astype(str))
    adata.var_names = pd.Index(syms.values)
    adata.var_names_make_unique()
    return sym_key

def split_complex(x):
    return [g for g in str(x).split("_") if g != ""]

def harmonize_complex_to_genes(complex_str, genes_set, looks_like_mouse):
    parts = split_complex(complex_str)
    out = []
    for p in parts:
        if p in genes_set:
            out.append(p)
            continue
        if looks_like_mouse:
            pm = human_to_mouse_symbol(p)
            if pm in genes_set:
                out.append(pm)
            else:
                out.append(p)
        else:
            # (optional) if you ever run human data with mouse-ish resource
            pu = str(p).upper()
            if pu in genes_set:
                out.append(pu)
            else:
                out.append(p)
    return "_".join(out)

def filter_resource_to_adata_genes(resource_df, adata_var_names, looks_like_mouse):
    """
    resource_df must have columns: ligand, receptor
    Returns resource with complexes harmonized + filtered to those fully present in adata genes.
    """
    genes = set(pd.Index(adata_var_names).astype(str))
    r = resource_df.copy()

    r["ligand"] = r["ligand"].astype(str).map(lambda x: harmonize_complex_to_genes(x, genes, looks_like_mouse))
    r["receptor"] = r["receptor"].astype(str).map(lambda x: harmonize_complex_to_genes(x, genes, looks_like_mouse))

    def all_in_genes(complex_str):
        return all((g in genes) for g in split_complex(complex_str))

    keep = r["ligand"].map(all_in_genes) & r["receptor"].map(all_in_genes)
    r = r.loc[keep].drop_duplicates().reset_index(drop=True)
    return r

def filter_obs_celltypes_timepoint(adata, celltype_key, sample_key, exclude_celltypes=("MC","Other"), min_cells=20):
    adx = adata.copy()
    if exclude_celltypes and celltype_key in adx.obs.columns:
        before = adx.n_obs
        adx = adx[~adx.obs[celltype_key].astype(str).isin(list(exclude_celltypes))].copy()
        print(f"[filter] excluded {list(exclude_celltypes)}: {before} -> {adx.n_obs}")

    adx.obs["celltype_timepoint"] = adx.obs[celltype_key].astype(str) + "_" + adx.obs[sample_key].astype(str)
    vc = adx.obs["celltype_timepoint"].value_counts()
    valid = vc[vc >= int(min_cells)].index
    before = adx.n_obs
    adx = adx[adx.obs["celltype_timepoint"].isin(valid)].copy()
    print(f"[filter] celltype_timepoint min_cells={min_cells}: {before} -> {adx.n_obs}")
    return adx

def normalize_log1p_inplace(adata):
    sc.pp.normalize_total(adata)
    sc.pp.log1p(adata)
    return adata


# -------------------------
# LIANA (protein) — FIXED for mouse
# -------------------------
def run_liana_protein_by_sample(
    adata,
    sample_key,
    groupby,
    looks_like_mouse,
    cache_csv: Path,
    expr_prop=0.1,
    min_cells=5,
    n_perms=25,              # safer default for big data; bump to 100 if stable
    return_all_lrs=True
):
    cache_csv = Path(cache_csv)
    if cache_csv.exists():
        print(f"[cache] protein CCC: {cache_csv}")
        return pd.read_csv(cache_csv)

    # Pick correct built-in resource name
    # (LIANA resources include 'mouseconsensus') :contentReference[oaicite:1]{index=1}
    resource_name = "mouseconsensus" if looks_like_mouse else "consensus"

    # Load as DataFrame and *filter to genes actually present* to avoid "100% missing"
    base_res = li.resource.select_resource(resource_name)
    prot_res = filter_resource_to_adata_genes(base_res, adata.var_names, looks_like_mouse)
    if prot_res.shape[0] == 0:
        raise RuntimeError(
            f"[protein resource] After filtering, 0 LR pairs remain using {resource_name}. "
            f"Likely gene IDs in adata.var_names still don't match expected symbols."
        )
    print(f"[protein resource] {resource_name}: kept {prot_res.shape[0]} LR pairs after gene filtering")

    # Run
    li.mt.rank_aggregate.by_sample(
        adata,
        sample_key=sample_key,
        groupby=groupby,
        resource=prot_res,          # <-- KEY: pass explicit resource
        expr_prop=float(expr_prop),
        min_cells=int(min_cells),
        n_perms=int(n_perms),
        use_raw=False,
        verbose=True,
        inplace=True,
        return_all_lrs=bool(return_all_lrs),
    )
    df = adata.uns["liana_res"].copy()
    df.to_csv(cache_csv, index=False)
    return df


# -------------------------
# Metalinks (as you had) — unchanged
# -------------------------
METALINKS_SOURCES = ["CellPhoneDB", "Cellinker", "scConnect", "recon", "hmr", "rhea", "hmdb"]
METALINKS_TYPES = ["pd", "lr"]

def prepare_metalinks_resources(adata_gene_names, biospecimen_location="Blood"):
    metalinks = li.resource.get_metalinks(
        biospecimen_location=biospecimen_location,
        source=METALINKS_SOURCES,
        types=METALINKS_TYPES,
    )

    resource_lr = (metalinks[metalinks["type"] == "lr"]
                   .loc[:, ["metabolite", "gene_symbol"]]
                   .rename(columns={"gene_symbol": "receptor"})
                   .drop_duplicates())

    pd_net = (metalinks[metalinks["type"] == "pd"]
              .loc[:, ["metabolite", "gene_symbol", "mor"]]
              .groupby(["metabolite", "gene_symbol"]).agg("mean").reset_index()
              .rename(columns={"metabolite": "source", "gene_symbol": "target", "mor": "weight"}))

    t_net = metalinks[metalinks["type"] == "pd"].loc[:, ["metabolite", "gene_symbol", "transport_direction"]].dropna()
    t_net["mor"] = t_net["transport_direction"].map(lambda x: 1 if x == "out" else (-1 if x == "in" else None))
    t_net = (t_net.loc[:, ["metabolite", "gene_symbol", "mor"]]
             .dropna()
             .groupby(["metabolite", "gene_symbol"]).agg("mean").reset_index())
    t_net = t_net[t_net["mor"] != 0].rename(columns={"metabolite": "source", "gene_symbol": "target", "mor": "weight"})

    # simple mouse-case harmonization (optional)
    genes = set(pd.Index(adata_gene_names).astype(str))
    def map_gene(g):
        g = str(g)
        if g in genes:
            return g
        gm = human_to_mouse_symbol(g)
        return gm if gm in genes else g

    resource_lr["receptor"] = resource_lr["receptor"].astype(str).map(map_gene)
    pd_net["target"] = pd_net["target"].astype(str).map(map_gene)
    t_net["target"] = t_net["target"].astype(str).map(map_gene)

    return resource_lr, pd_net, t_net

def estimate_metabolites_mudata_robust(adata, resource_lr, pd_net, t_net, groupby, sample_key):
    dc.mt.ulm(adata, net=pd_net, raw=False)
    met_est = adata.obsm["score_ulm"].copy()

    dc.mt.waggr(adata, t_net, times=0, raw=False, fun="wmean")
    out_est = adata.obsm["score_waggr"].copy()

    intersect = np.intersect1d(met_est.columns, out_est.columns)
    out_est = out_est[intersect]
    mask = (out_est > 0).astype(float).to_numpy()
    mmat = met_est[intersect] * mask
    coldiff = np.setdiff1d(met_est.columns, mmat.columns)
    if len(coldiff) > 0:
        mmat = pd.concat([mmat, met_est[coldiff]], axis=1)

    _resource = resource_lr[resource_lr["metabolite"].isin(mmat.columns)].copy()
    recs = pd.Index(_resource["receptor"].astype(str).unique())
    recs_in = recs.intersection(pd.Index(adata.var_names.astype(str)))
    receptor = adata[:, recs_in].copy()

    adata.obsm["mmat"] = mmat
    mmat_adata = li.utils.obsm_to_adata(adata, "mmat")
    meta = MuData({"metabolite": mmat_adata, "receptor": receptor})

    meta.obs[groupby] = adata.obs[groupby].astype("category")
    meta.obs[sample_key] = adata.obs[sample_key].astype("category")
    return meta

def run_liana_metabolite_by_sample(meta, resource_lr, sample_key, groupby, cache_csv: Path,
                                   expr_prop=0.1, min_cells=5, n_perms=25):
    cache_csv = Path(cache_csv)
    if cache_csv.exists():
        print(f"[cache] metabolite CCC: {cache_csv}")
        return pd.read_csv(cache_csv)

    li.mt.rank_aggregate.by_sample(
        meta,
        sample_key=sample_key,
        groupby=groupby,
        use_raw=False,
        resource=resource_lr.rename(columns={"metabolite": "ligand"}),
        mdata_kwargs={
            "x_mod": "metabolite",
            "y_mod": "receptor",
            "x_use_raw": False,
            "y_use_raw": False,
            "x_transform": li.ut.zi_minmax,
            "y_transform": li.ut.zi_minmax,
        },
        expr_prop=float(expr_prop),
        min_cells=int(min_cells),
        n_perms=int(n_perms),
        verbose=True,
    )
    df = meta.uns["liana_res"].copy()
    df.to_csv(cache_csv, index=False)
    return df


# -------------------------
# Backed-mode safe load slice (Fix A)
# -------------------------
def load_slice_to_memory(rnaseq_backed, cell_ids, var_mask):
    # IMPORTANT: in backed mode, .copy() without filename fails; use .to_memory()
    view = rnaseq_backed[cell_ids, var_mask]
    adx = view.to_memory()  # <-- FIX
    return adx


# ============================================================
# RUN (minimal: only until protein CCC is fixed)
# ============================================================

OUTDIR = Path("liana_coupled_tensor_sex_outputs")
OUTDIR.mkdir(parents=True, exist_ok=True)

H5AD_PATH = Path("Soni_Brain_2025_cellxgene_5a6c74b9-da1c-4cef-8fdc-07c7a90089d7.h5ad")
print(f"[load] reading: {H5AD_PATH}")
rnaseq = sc.read_h5ad(str(H5AD_PATH), backed="r")  # backed to avoid kernel crash

CELLTYPE_CANDIDATES = ["celltype", "cell_type", "Celltype", "annotation"]
SAMPLE_CANDIDATES   = ["orig.ident", "sample_id", "sample", "timepoint", "batch"]
SEX_CANDIDATES      = ["sex", "Sex", "gender", "Gender"]

celltype_key = detect_obs_key(rnaseq, CELLTYPE_CANDIDATES, label="celltype_key")
sample_key   = detect_obs_key(rnaseq, SAMPLE_CANDIDATES,   label="sample_key")
sex_key      = detect_obs_key(rnaseq, SEX_CANDIDATES,      label="sex_key")

print("[Detected keys]")
print(" celltype_key =", celltype_key)
print(" sample_key   =", sample_key)
print(" sex_key      =", sex_key)

# Heuristic species check uses current var_names (may be Ensembl); better use gene_symbols if present
sym_key = detect_var_symbol_key(rnaseq)
if sym_key is not None:
    genes_for_heur = rnaseq.var[sym_key].astype(str).values[:5000]
else:
    genes_for_heur = rnaseq.var_names.astype(str).values[:5000]
looks_mouse = looks_like_mouse_symbols(genes_for_heur)
print("[species heuristic] looks_like_mouse =", looks_mouse)

# Build metalinks now (for later)
# NOTE: we will set var_names to gene symbols AFTER slicing to memory (cheaper)
# but for metalinks filtering we need the eventual gene symbols; we’ll use var[sym_key] if present
if sym_key is not None:
    adata_gene_symbols_all = rnaseq.var[sym_key].astype(str).values
else:
    adata_gene_symbols_all = rnaseq.var_names.astype(str).values

resource_lr, pd_net, t_net = prepare_metalinks_resources(adata_gene_symbols_all, biospecimen_location="Blood")
print("[metalinks] LR rows:", resource_lr.shape, "| PD rows:", pd_net.shape, "| T rows:", t_net.shape)

# --- Choose genes to keep (important for memory)
# Use correct protein resource for mouse vs human
protein_res_name = "mouseconsensus" if looks_mouse else "consensus"
prot_res_df = li.resource.select_resource(protein_res_name)  # ligand/receptor columns :contentReference[oaicite:2]{index=2}
prot_genes = set()
for c in prot_res_df["ligand"].astype(str).tolist() + prot_res_df["receptor"].astype(str).tolist():
    for g in split_complex(c):
        prot_genes.add(g)

metalinks_genes = set(resource_lr["receptor"].astype(str).tolist())
keep_genes = prot_genes.union(metalinks_genes)
print(f"[genes] keep set size ≈ {len(keep_genes)} (protein={len(prot_genes)}, metalinks={len(metalinks_genes)})")

# var_mask based on gene symbols if available, else var_names
if sym_key is not None:
    var_syms = rnaseq.var[sym_key].astype(str)
    var_mask = var_syms.isin(list(keep_genes)).to_numpy()
else:
    var_mask = rnaseq.var_names.astype(str).isin(list(keep_genes)).to_numpy()

print(f"[genes] var_mask keeps {int(var_mask.sum())} genes out of {rnaseq.n_vars}")

# --- Cell filtering (cheap, obs only) + optional downsample
obs = rnaseq.obs
mask_ct = ~obs[celltype_key].astype(str).isin(["MC", "Other"])
obs2 = obs.loc[mask_ct].copy()
# keep only celltype_timepoint >= 20
ctp = obs2[celltype_key].astype(str) + "_" + obs2[sample_key].astype(str)
vc = ctp.value_counts()
valid = set(vc[vc >= 20].index)
cell_ids = obs2.index[ctp.isin(valid)]
print(f"[all] cells kept after obs-filter: {len(cell_ids)}")

# --- Load slice to memory (FIX for backed copy)
adx = load_slice_to_memory(rnaseq, cell_ids, var_mask)
ensure_sparse_float32(adx)

# Set var_names to gene symbols now (in-memory)
sym_key2 = make_gene_symbol_view_inplace(adx)
if sym_key2 is not None:
    print(f"[genes] using gene symbols from var['{sym_key2}'] for resource matching")

# Final normalize
adx = filter_obs_celltypes_timepoint(adx, celltype_key, sample_key, exclude_celltypes=None, min_cells=20)
normalize_log1p_inplace(adx)

# --- Run protein CCC (FIXED)
protein_ccc = run_liana_protein_by_sample(
    adx,
    sample_key=sample_key,
    groupby=celltype_key,
    looks_like_mouse=looks_mouse,
    cache_csv=OUTDIR / "all" / "protein_liana_res.csv",
    expr_prop=0.1,
    min_cells=5,
    n_perms=25,          # increase to 100 once stable
    return_all_lrs=True
)

print(protein_ccc.head())
print("[OK] protein CCC computed.")


In [ ]:
# ============================================================
# DOWNSTREAM PLOTS: Tensor build + (Coupled) CTCA + plotting
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import liana as li
import cell2cell as c2c


# -------------------------
# Configure where your cached CSVs live
# -------------------------
# If you used OUTDIR / "all" / "protein_liana_res.csv" earlier, point to that root:
BASE = Path("liana_coupled_tensor_sex_outputs")   # <-- change if needed
# e.g. BASE = Path("/home/aalentorn/Téléchargements/liana_coupled_tensor_sex_outputs")

# subset folders that you may have (any of these can exist)
SUBSETS = ["all", "female", "male"]

# plotting output folder
PLOT_DIR = BASE / "DOWNSTREAM_PLOTS"
PLOT_DIR.mkdir(parents=True, exist_ok=True)


# -------------------------
# Helpers
# -------------------------
def _maybe_read_csv(path: Path):
    return pd.read_csv(path) if path.exists() else None

def choose_score_key(df: pd.DataFrame):
    """
    Pick a robust score column for tensor building.
    Preference: lrscore (0..1), then scaled_weight/expr_prod/lr_means, else magnitude_rank (rank-like).
    """
    for k in ["lrscore", "scaled_weight", "expr_prod", "lr_means"]:
        if k in df.columns:
            return k, None  # no inverse
    if "magnitude_rank" in df.columns:
        # magnitude_rank is often "rank-ish": smaller can mean stronger
        # turn it into a "higher is better" score:
        return "magnitude_rank", (lambda x: 1 - x)
    # fallback to first numeric
    num = df.select_dtypes(include=[np.number]).columns.tolist()
    if len(num) == 0:
        raise ValueError("No numeric score column found in LIANA result.")
    return num[0], None

def build_tensor_from_liana(df, sample_key, contexts, lri_label="LRIs",
                           outer_fraction=1/3):
    score_key, inverse_fun = choose_score_key(df)

    tensor = li.multi.to_tensor_c2c(
        liana_res=df,
        sample_key=sample_key,
        source_key="source",
        target_key="target",
        ligand_key="ligand_complex",
        receptor_key="receptor_complex",
        score_key=score_key,
        inverse_fun=inverse_fun,
        how="outer_cells",
        outer_fraction=float(outer_fraction),
        context_order=contexts,
        order_labels=["Contexts", lri_label, "Sender Cells", "Receiver Cells"],
    )
    print(f"[tensor] built with score_key='{score_key}' | shape={tensor.shape}")
    return tensor

def plot_tensor_suite(interaction_tensor, outdir: Path,
                      lri_dim_name="LRIs",
                      rank=6,
                      random_state=888,
                      cci_threshold=0.2,
                      lr_threshold=0.15):
    """
    Works for either InteractionTensor or CoupledInteractionTensor.
    Saves:
      - Tensor-Factorization.pdf
      - Networks.svg
      - Clustermap-Factor-specific-CCI.pdf
      - Clustermap-Factor-specific-LRs.pdf
      - Loadings.xlsx
    """
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    # Factorization
    interaction_tensor.compute_tensor_factorization(
        rank=int(rank),
        init="svd",
        random_state=int(random_state),
        runs=1,
        tol=1e-8,
        n_iter_max=500,
    )

    # Export loadings
    interaction_tensor.export_factor_loadings(str(outdir / "Loadings.xlsx"))

    # --- Factor plot (metadata)
    # Build minimal metadata so it labels the LRI mode as a category
    if isinstance(interaction_tensor, c2c.tensor.CoupledInteractionTensor):
        # In coupled, we can create metadata per tensor and reorder
        # Label protein LRIs and metabolite LRIs separately if present
        md_list = []
        for dim in interaction_tensor.order_names:
            if dim == "Protein LRIs":
                md_list.append({k: "Proteins" for k in interaction_tensor.order_names[dim]})
            elif dim == "Metabolite LRIs":
                md_list.append({k: "Metabolites" for k in interaction_tensor.order_names[dim]})
            else:
                md_list.append(None)

        # tensor_factors_plot expects a metadata dataframe; easiest is to let cell2cell fill labels:
        meta = c2c.tensor.generate_tensor_metadata(
            interaction_tensor=interaction_tensor,
            metadata_dicts=md_list,
            fill_with_order_elements=True,
        )
    else:
        # protein-only tensor
        lri_dim = lri_dim_name
        md = []
        for dim in interaction_tensor.order_names:
            if dim == lri_dim:
                md.append({k: "LRIs" for k in interaction_tensor.order_names[dim]})
            else:
                md.append(None)
        meta = c2c.tensor.generate_tensor_metadata(
            interaction_tensor=interaction_tensor,
            metadata_dicts=md,
            fill_with_order_elements=True,
        )

    c2c.plotting.tensor_factors_plot(
        interaction_tensor=interaction_tensor,
        metadata=meta,
        sample_col="Element",
        group_col="Category",
        meta_cmaps=["viridis", "tab20", "tab20", "Dark2", "Dark2"],
        order_sorting=["Contexts", "Sender Cells", "Receiver Cells", "Protein LRIs", "Metabolite LRIs", "LRIs"],
        fontsize=14,
        filename=str(outdir / "Tensor-Factorization.pdf"),
    )

    # --- Factor-specific CCI networks (sender/receiver)
    networks = c2c.analysis.tensor_downstream.get_factor_specific_ccc_networks(
        interaction_tensor.factors,
        sender_label="Sender Cells",
        receiver_label="Receiver Cells",
    )
    network_by_factors = c2c.analysis.tensor_downstream.flatten_factor_ccc_networks(networks, orderby="receivers")
    factor_sorted = [f"Factor {f}" for f in range(1, interaction_tensor.rank + 1)]

    c2c.plotting.ccc_networks_plot(
        interaction_tensor.factors,
        included_factors=None,
        ccc_threshold=float(cci_threshold),
        nrows=2,
        panel_size=(8, 8),
        network_layout="circular",
        filename=str(outdir / "Networks.svg"),
    )

    # CCI clustermap
    c2c.plotting.loading_clustermap(
        network_by_factors[factor_sorted],
        use_zscore=False,
        loading_threshold=float(cci_threshold),
        figsize=(20, 9),
        tick_fontsize=16,
        cmap="YlOrBr",
        filename=str(outdir / "Clustermap-Factor-specific-CCI.pdf"),
        row_cluster=False,
    )

    # LR clustermap(s)
    # For coupled, we concatenate Protein LRIs + Metabolite LRIs if present
    if isinstance(interaction_tensor, c2c.tensor.CoupledInteractionTensor):
        pieces = []
        for dim in ["Protein LRIs", "Metabolite LRIs"]:
            if dim in interaction_tensor.factors:
                df = interaction_tensor.factors[dim].copy()
                df = df[(df.T > float(lr_threshold)).any()]
                if df.shape[0] > 0:
                    df = (df / (df.max() + 1e-12))
                    pieces.append(df)
        if len(pieces) > 0:
            df_lr = pd.concat(pieces, axis=0)
            c2c.plotting.loading_clustermap(
                df_lr,
                loading_threshold=float(lr_threshold),
                use_zscore=False,
                figsize=(24, 6),
                cbar_fontsize=20,
                tick_fontsize=9,
                cmap="YlOrBr",
                filename=str(outdir / "Clustermap-Factor-specific-LRs.pdf"),
                row_cluster=False,
            )
    else:
        if lri_dim_name in interaction_tensor.factors:
            df = interaction_tensor.factors[lri_dim_name].copy()
            df = df[(df.T > float(lr_threshold)).any()]
            if df.shape[0] > 0:
                df = (df / (df.max() + 1e-12))
                c2c.plotting.loading_clustermap(
                    df,
                    loading_threshold=float(lr_threshold),
                    use_zscore=False,
                    figsize=(20, 6),
                    cbar_fontsize=20,
                    tick_fontsize=9,
                    cmap="YlOrBr",
                    filename=str(outdir / "Clustermap-Factor-specific-LRs.pdf"),
                    row_cluster=False,
                )

    print("[plots] saved in:", outdir.resolve())

def factor_similarity_heatmap(coupled_a, coupled_b, outpath, which="Sender Cells"):
    A = coupled_a.factors[which].copy()
    B = coupled_b.factors[which].copy()
    A = A / (A.max(axis=0) + 1e-12)
    B = B / (B.max(axis=0) + 1e-12)
    idx = A.index.intersection(B.index)
    A = A.loc[idx]
    B = B.loc[idx]
    corr = np.corrcoef(A.T, B.T)[:A.shape[1], A.shape[1]:]
    corr = pd.DataFrame(corr, index=A.columns, columns=B.columns)

    plt.figure(figsize=(7, 6))
    plt.imshow(corr.values, aspect="auto")
    plt.xticks(range(corr.shape[1]), corr.columns, rotation=90)
    plt.yticks(range(corr.shape[0]), corr.index)
    plt.colorbar(label="corr")
    plt.title(f"Factor similarity — {which}")
    plt.tight_layout()
    plt.savefig(outpath, dpi=200)
    plt.close()
    return corr


# -------------------------
# Main: load cached LIANA results and plot
# -------------------------
all_objs = {}  # store per subset for optional sex comparison

for subset in SUBSETS:
    prot_path = BASE / subset / "protein_liana_res.csv"
    met_path  = BASE / subset / "metabolite_liana_res.csv"

    protein_ccc = _maybe_read_csv(prot_path)
    if protein_ccc is None:
        print(f"[skip] {subset}: missing {prot_path}")
        continue

    # infer keys
    sample_key = "sample_id" if "sample_id" in protein_ccc.columns else "sample"
    contexts = sorted(protein_ccc[sample_key].astype(str).unique().tolist())

    print("\n" + "="*20, subset.upper(), "="*20)
    print("[load] protein:", prot_path, "| rows=", protein_ccc.shape[0])
    print("[contexts]", contexts)

    # build protein tensor
    tensor_prot = build_tensor_from_liana(
        protein_ccc, sample_key=sample_key, contexts=contexts, lri_label="Protein LRIs"
    )

    # try coupled if metabolite exists
    met_ccc = _maybe_read_csv(met_path)
    if met_ccc is not None and met_ccc.shape[0] > 0:
        print("[load] metabolite:", met_path, "| rows=", met_ccc.shape[0])

        tensor_met = build_tensor_from_liana(
            met_ccc, sample_key=sample_key, contexts=contexts, lri_label="Metabolite LRIs"
        )

        coupled = c2c.tensor.CoupledInteractionTensor(
            tensor1=tensor_prot,
            tensor2=tensor_met,
            tensor1_name="protein",
            tensor2_name="metabolite",
            mode_mapping={"shared": [(0, 0), (2, 2), (3, 3)]},  # contexts + sender + receiver shared
        )

        outdir = PLOT_DIR / subset / "COUPLED_CTCA"
        plot_tensor_suite(
            interaction_tensor=coupled,
            outdir=outdir,
            rank=6,
            random_state=888,
            cci_threshold=0.2,
            lr_threshold=0.15
        )
        all_objs[subset] = {"coupled": coupled}

    else:
        # protein-only
        outdir = PLOT_DIR / subset / "PROTEIN_ONLY"
        plot_tensor_suite(
            interaction_tensor=tensor_prot,
            outdir=outdir,
            lri_dim_name="Protein LRIs",
            rank=6,
            random_state=888,
            cci_threshold=0.2,
            lr_threshold=0.15
        )


# -------------------------
# Optional: female vs male factor similarity (only if both coupled exist)
# -------------------------
if ("female" in all_objs) and ("male" in all_objs):
    cf = all_objs["female"]["coupled"]
    cm = all_objs["male"]["coupled"]
    for dim in ["Contexts", "Sender Cells", "Receiver Cells"]:
        corr = factor_similarity_heatmap(
            cf, cm,
            outpath=PLOT_DIR / f"female_vs_male_factor_similarity_{dim.replace(' ','_')}.png",
            which=dim
        )
        corr.to_csv(PLOT_DIR / f"female_vs_male_factor_similarity_{dim.replace(' ','_')}.csv")
    print("[sex-compare] saved in:", PLOT_DIR.resolve())

print("\n[DONE] Downstream plots in:", PLOT_DIR.resolve())
